# A Big-Data Pipeline for Bank Failure Early Warning
**Course:** BFT65B – Big Data Analytics in Finance (Final Individual Research Report)  
**Student:** Nguyễn Bá Dương | **ID:** 11230945  
**GitHub:** [Link GitHub public của bạn]

---
## Notebook Overview
This notebook implements an end-to-end, reproducible Big Data Engineering pipeline using **PyArrow, Polars, Pandas, PySpark, and scikit-learn**. It transforms 42 years of raw U.S. FDIC Call Reports (`1984Q1–2026Q2`, ~1.68 million rows) into a leak-free early-warning predictive system. The pipeline is strictly divided into 5 chronological parts to prevent data leakage and ensure memory-efficient execution.

## Environment Setup & Drive Mounting
**Objective:** Mount the Google Drive virtual filesystem to persist Bronze/Silver/Gold datasets securely and manage OS-level environment variables.
Additionally, this section dynamically verifies and installs strict Python library versions (pinned via `requirements.txt` specifications) to guarantee exact reproducibility across distributed and single-node compute engines without redundant installations.

In [2]:
# Mount Google Drive và thiết lập thư mục
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

# Đường dẫn lưu dữ liệu trên Google Drive (Tồn tại lâu dài, không mất khi tắt Colab)
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/BFT65B_Project/data_raw/fdic")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/BFT65B_Project/outputs")
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Truyền vào biến môi trường để các script Python tự động nhận diện
os.environ["FDIC_DATA_DIR"] = str(DRIVE_DATA_DIR)
os.environ["FDIC_OUTPUT_DIR"] = str(DRIVE_OUTPUT_DIR)

print(f"Data directory set to: {os.environ['FDIC_DATA_DIR']}")

Mounted at /content/drive
Data directory set to: /content/drive/MyDrive/BFT65B_Project/data_raw/fdic


In [17]:
# Smart Dependency Verification and Installation
import importlib.metadata
import subprocess
import sys

# Define strict target versions matching requirements.txt
REQUIRED_PACKAGES = {
    "requests": "2.32.3",
    "pandas": "2.2.2",
    "pyarrow": "17.0.0",
    "polars": "1.9.0",
    "pyspark": "3.5.3",
    "scikit-learn": "1.5.2",
    "joblib": "1.4.2",
    "psutil": "6.0.0"
}

def check_and_install_packages(requirements: dict):
    packages_to_install = []
    for package, target_version in requirements.items():
        try:
            installed_version = importlib.metadata.version(package)
            print(f"[OK] {package} is already installed (Version: {installed_version}).")

            # (Optional) If you strictly want to force exact version match, uncomment below:
            # if installed_version != target_version:
            #     packages_to_install.append(f"{package}=={target_version}")

        except importlib.metadata.PackageNotFoundError:
            print(f"[MISSING] {package} not found. Flagged for installation.")
            packages_to_install.append(f"{package}=={target_version}")

    if packages_to_install:
        install_cmd = " ".join(packages_to_install)
        print(f"\nInstalling missing packages: {install_cmd} ...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q"] + packages_to_install
        )
        print("Installation complete!")
    else:
        print("\nAll required packages are already installed. Ready to proceed!")

# Execute the check
check_and_install_packages(REQUIRED_PACKAGES)

[OK] requests is already installed (Version: 2.32.4).
[OK] pandas is already installed (Version: 2.2.3).
[OK] pyarrow is already installed (Version: 23.0.1).
[OK] polars is already installed (Version: 1.35.2).
[OK] pyspark is already installed (Version: 4.0.5).
[OK] scikit-learn is already installed (Version: 1.6.1).
[OK] joblib is already installed (Version: 1.6.0).
[OK] psutil is already installed (Version: 5.9.5).

All required packages are already installed. Ready to proceed!


## Part 1: Data Acquisition & Storage Architecture
**Objective:** Ingest four decades of supervisory data securely, enforce strict types to prevent historical schema drift, and empirically justify the physical storage layout.

**Sections:**
1. **Environment Setup:** Mounts the virtual filesystem and pins exact library versions for reproducibility.
2. **Resumable API Ingestion (`src/p1_acquire.py`):** Fetches the `/financials`, `/failures`, and `/institutions` tables via the FDIC REST API. It implements exponential backoff and enforces a static PyArrow schema (`FIN_SCHEMA`) to prevent `SchemaMismatchError` caused by pre-Basel I (1984–1989) missing capital ratios.
3. **Storage Benchmark (`src/p1_storage_benchmark.py`):** Measures disk footprint, write latency, projection pushdown, and predicate filtering speeds across Raw CSV, Uncompressed Parquet, Snappy, ZSTD, and Quarter-Sharded Snappy layouts.

**Pipeline Linkage:** This part materializes the **Bronze Layer** (`data_raw/fdic/`), which serves as the immutable single-source-of-truth for the data quality audit in Part 2.

In [4]:
"""
p1_acquire.py
Acquires FDIC Call Report financials (1984Q1 - 2026Q2), failures, and institutions
with explicit PyArrow schema enforcement and completeness verification.
Adapted from course starter script fdic_download_starter.py.
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time
from datetime import date
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests

BASE = "https://api.fdic.gov/banks"
PAGE_SIZE = 10_000

FIN_FIELDS = [
    "CERT", "REPDTE", "STNAME",
    "ASSET", "DEP", "DEPDOM", "EQ", "BRO", "CHBAL", "SC",
    "LNLSGR", "LNLSNET", "LNRE", "LNCI", "LNCON", "LNATRES",
    "NCLNLS", "P3ASSET", "P9ASSET", "NAASSET", "NTLNLS", "ORE",
    "NETINC", "INTINC", "EINTEXP", "NONII", "NONIX", "ELNATR",
    "ROA", "ROE", "NIMY",
    "RBC1AAJ", "RBCRWAJ",
]
FAIL_FIELDS = ["CERT", "NAME", "CITYST", "FAILDATE", "FAILYR", "COST", "QBFASSET", "QBFDEP", "RESTYPE"]
INST_FIELDS = ["CERT", "NAME", "STNAME", "BKCLASS", "ACTIVE", "ESTYMD", "ENDEFYMD", "CHANGEC1", "INSDATE"]

# Explicit PyArrow Schema for Financials to prevent schema drift across 1984-2026
FIN_NUMERIC_COLS = [c for c in FIN_FIELDS if c not in ("CERT", "REPDTE", "STNAME")]
FIN_SCHEMA = pa.schema(
    [
        pa.field("CERT", pa.int64()),
        pa.field("REPDTE", pa.string()),
        pa.field("STNAME", pa.string()),
    ]
    + [pa.field(col, pa.float64()) for col in FIN_NUMERIC_COLS]
)

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "BFT65B-DataPipeline/1.0"})


def get_json(endpoint: str, params: dict, max_retries: int = 6, pause: float = 0.8) -> dict:
    url = f"{BASE}/{endpoint}"
    for attempt in range(1, max_retries + 1):
        try:
            r = SESSION.get(url, params=params, timeout=180)
            if r.status_code == 200:
                time.sleep(pause)
                return r.json()
            print(f"    HTTP {r.status_code} (attempt {attempt}) - retrying")
        except requests.RequestException as exc:
            print(f"    {type(exc).__name__} (attempt {attempt}) - retrying")
        time.sleep(min(60, 2 ** attempt))
    raise RuntimeError(f"Failed GET {url} with params {params}")


def fetch_all(endpoint: str, fields: list[str], filters: str | None = None,
              sort_by: str = "CERT") -> tuple[pd.DataFrame, int]:
    rows, offset, total = [], 0, None
    while True:
        params = {
            "fields": ",".join(fields),
            "limit": PAGE_SIZE,
            "offset": offset,
            "sort_by": sort_by,
            "sort_order": "ASC",
            "format": "json",
        }
        if filters:
            params["filters"] = filters
        js = get_json(endpoint, params)
        total = js["meta"]["total"]
        page = [rec["data"] for rec in js["data"]]
        rows.extend(page)
        offset += len(page)
        if not page or offset >= total:
            break
    df = pd.DataFrame(rows)
    return df.drop(columns=["ID"], errors="ignore"), total


def enforce_fin_schema(df: pd.DataFrame) -> pa.Table:
    """Ensure all expected columns exist with consistent types across all quarters."""
    for col in FIN_FIELDS:
        if col not in df.columns:
            df[col] = None
    df = df[FIN_FIELDS].copy()
    df["CERT"] = pd.to_numeric(df["CERT"], errors="coerce").astype("Int64")
    df["REPDTE"] = df["REPDTE"].astype(str)
    df["STNAME"] = df["STNAME"].astype("string")
    for col in FIN_NUMERIC_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")
    return pa.Table.from_pandas(df, schema=FIN_SCHEMA, preserve_index=False)


def quarter_ends(start: date, end: date) -> list[str]:
    ends = []
    for y in range(start.year, end.year + 1):
        for m, d in ((3, 31), (6, 30), (9, 30), (12, 31)):
            q = date(y, m, d)
            if start <= q <= end:
                ends.append(q.strftime("%Y%m%d"))
    return ends


def download_financials(out: Path, quarters: list[str], compression: str = "snappy") -> None:
    fin_dir = out / "financials"
    fin_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = out / "manifest_financials.json"
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}

    for i, q in enumerate(quarters, 1):
        target = fin_dir / f"REPDTE={q}.parquet"
        if target.exists() and manifest.get(q, {}).get("complete"):
            print(f"[{i}/{len(quarters)}] {q}: already complete, skipped")
            continue
        t0 = time.time()
        df, total = fetch_all("financials", FIN_FIELDS, filters=f"REPDTE:{q}")
        if df.empty:
            print(f"[{i}/{len(quarters)}] {q}: no data returned")
            continue
        table = enforce_fin_schema(df)
        pq.write_table(table, target, compression=compression)
        complete = (len(df) == total) and (not df.duplicated(["CERT", "REPDTE"]).any())
        manifest[q] = {
            "rows": len(df),
            "api_total": total,
            "complete": bool(complete),
            "size_bytes": target.stat().st_size,
        }
        manifest_path.write_text(json.dumps(manifest, indent=2))
        print(f"[{i}/{len(quarters)}] {q}: {len(df):>6,} rows (API {total:,}) "
              f"{'OK' if complete else 'INCOMPLETE'} | {time.time() - t0:5.1f}s")


def download_small_tables(out: Path, compression: str = "snappy") -> None:
    for name, fields, sort_by in (
        ("failures", FAIL_FIELDS, "FAILDATE"),
        ("institutions", INST_FIELDS, "CERT"),
    ):
        t0 = time.time()
        df, total = fetch_all(name, fields, sort_by=sort_by)
        if "CERT" in df.columns:
            df["CERT"] = pd.to_numeric(df["CERT"], errors="coerce").astype("Int64")
        target = out / f"{name}.parquet"
        df.to_parquet(target, index=False, compression=compression)
        print(f"{name:<12}: {len(df):>6,} rows (API {total:,}) | {time.time() - t0:5.1f}s")


def verify_completeness(out: Path) -> dict:
    manifest = json.loads((out / "manifest_financials.json").read_text())
    panel = pd.read_parquet(out / "financials", columns=["CERT", "REPDTE"])
    fails = pd.read_parquet(out / "failures.parquet")
    insts = pd.read_parquet(out / "institutions.parquet")

    n_dup = int(panel.duplicated(["CERT", "REPDTE"]).sum())
    incomplete_q = [q for q, m in manifest.items() if not m["complete"]]
    api_row_sum = sum(m["api_total"] for m in manifest.values())

    summary = {
        "quarters_downloaded": len(manifest),
        "first_quarter": min(manifest.keys()),
        "last_quarter": max(manifest.keys()),
        "total_panel_rows": len(panel),
        "api_reported_rows": api_row_sum,
        "distinct_banks_panel": int(panel["CERT"].nunique()),
        "duplicate_cert_repdte": n_dup,
        "incomplete_quarters": incomplete_q,
        "failures_rows": len(fails),
        "institutions_rows": len(insts),
    }
    print("\n=== DATA COMPLETENESS SUMMARY ===")
    for k, v in summary.items():
        print(f"{k:<25}: {v}")
    return summary


if __name__ == "__main__":
    default_out = os.environ.get("FDIC_DATA_DIR", "data_raw/fdic")
    ap = argparse.ArgumentParser()
    ap.add_argument("--start", default="1984-03-31")
    ap.add_argument("--end", default="2026-06-30")
    ap.add_argument("--out", default=default_out)
    args, _ = ap.parse_known_args()

    out_path = Path(args.out)
    out_path.mkdir(parents=True, exist_ok=True)
    q_list = quarter_ends(date.fromisoformat(args.start), date.fromisoformat(args.end))
    print(f"Target quarters: {len(q_list)} ({q_list[0]} -> {q_list[-1]}) | Output: {out_path.resolve()}\n")

    download_financials(out_path, q_list)
    download_small_tables(out_path)
    verify_completeness(out_path)

Target quarters: 170 (19840331 -> 20260630) | Output: /content/drive/MyDrive/BFT65B_Project/data_raw/fdic

[1/170] 19840331: already complete, skipped
[2/170] 19840630: already complete, skipped
[3/170] 19840930: already complete, skipped
[4/170] 19841231: already complete, skipped
[5/170] 19850331: already complete, skipped
[6/170] 19850630: already complete, skipped
[7/170] 19850930: already complete, skipped
[8/170] 19851231: already complete, skipped
[9/170] 19860331: already complete, skipped
[10/170] 19860630: already complete, skipped
[11/170] 19860930: already complete, skipped
[12/170] 19861231: already complete, skipped
[13/170] 19870331: already complete, skipped
[14/170] 19870630: already complete, skipped
[15/170] 19870930: already complete, skipped
[16/170] 19871231: already complete, skipped
[17/170] 19880331: already complete, skipped
[18/170] 19880630: already complete, skipped
[19/170] 19880930: already complete, skipped
[20/170] 19881231: already complete, skipped
[2

In [5]:
"""
p1_storage_benchmark.py
Benchmarks storage layouts (Raw CSV vs. Parquet Uncompressed/Snappy/ZSTD vs. Partitioned Parquet)
across disk size, write throughput, full scan, column projection, and predicate pushdown.
"""
from __future__ import annotations

import gc
import os
import shutil
import time
from pathlib import Path

import pandas as pd
import polars as pl
import pyarrow.dataset as ds
import pyarrow.parquet as pq

PROJ_COLS = ["CERT", "REPDTE", "ASSET", "EQ", "ROA"]
FILTER_START = "20080331"
FILTER_END = "20101231"


def dir_size_mb(path: Path) -> float:
    if path.is_file():
        return path.stat().st_size / (1024 * 1024)
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / (1024 * 1024)


def time_op(fn, repeats: int = 3) -> float:
    """Run function `repeats` times, return median elapsed time in seconds."""
    times = []
    for _ in range(repeats):
        gc.collect()
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    times.sort()
    return round(times[len(times) // 2], 4)


def run_storage_benchmark(source_fin_dir: Path, work_dir: Path, out_dir: Path) -> pd.DataFrame:
    work_dir.mkdir(parents=True, exist_ok=True)
    out_dir.mkdir(parents=True, exist_ok=True)

    print("1. Loading partitioned dataset into memory to prepare benchmark baselines...")
    # Copy partitioned directory to local SSD if not already local
    local_part_dir = work_dir / "partitioned_snappy"
    if local_part_dir.exists():
        shutil.rmtree(local_part_dir)
    shutil.copytree(source_fin_dir, local_part_dir)

    # Read full table via PyArrow
    full_table = pq.read_table(local_part_dir)
    df_pandas = full_table.to_pandas()
    print(f"   Loaded {full_table.num_rows:,} rows x {full_table.num_columns} columns.")

    results = []

    # -------------------------------------------------------------------------
    # Layout 1: Raw CSV
    # -------------------------------------------------------------------------
    csv_path = work_dir / "financials_raw.csv"
    print("2. Benchmarking Raw CSV...")
    t_write_csv = time_op(lambda: df_pandas.to_csv(csv_path, index=False), repeats=1)
    size_csv = dir_size_mb(csv_path)

    t_full_csv = time_op(lambda: pl.read_csv(csv_path))
    t_proj_csv = time_op(lambda: pl.read_csv(csv_path, columns=PROJ_COLS))
    t_filt_csv = time_op(
        lambda: pl.scan_csv(csv_path)
        .filter((pl.col("REPDTE").cast(pl.Utf8) >= FILTER_START) & (pl.col("REPDTE").cast(pl.Utf8) <= FILTER_END))
        .select(PROJ_COLS)
        .collect()
    )

    results.append({
        "Layout": "Raw CSV (Single File)",
        "Compression": "None",
        "Size_MB": round(size_csv, 2),
        "Write_s": t_write_csv,
        "Read_Full_s": t_full_csv,
        "Read_Proj_5Cols_s": t_proj_csv,
        "Read_Filter_2008_2010_s": t_filt_csv,
    })

    # -------------------------------------------------------------------------
    # Layouts 2, 3, 4: Single Parquet (Uncompressed, Snappy, ZSTD)
    # -------------------------------------------------------------------------
    for comp in ("NONE", "SNAPPY", "ZSTD"):
        pq_path = work_dir / f"financials_{comp.lower()}.parquet"
        print(f"3. Benchmarking Single Parquet ({comp})...")
        t_write_pq = time_op(lambda: pq.write_table(full_table, pq_path, compression=comp), repeats=3)
        size_pq = dir_size_mb(pq_path)

        t_full_pq = time_op(lambda: pl.read_parquet(pq_path))
        t_proj_pq = time_op(lambda: pl.read_parquet(pq_path, columns=PROJ_COLS))
        t_filt_pq = time_op(
            lambda: pl.scan_parquet(pq_path)
            .filter((pl.col("REPDTE") >= FILTER_START) & (pl.col("REPDTE") <= FILTER_END))
            .select(PROJ_COLS)
            .collect()
        )

        results.append({
            "Layout": "Single Parquet",
            "Compression": comp,
            "Size_MB": round(size_pq, 2),
            "Write_s": t_write_pq,
            "Read_Full_s": t_full_pq,
            "Read_Proj_5Cols_s": t_proj_pq,
            "Read_Filter_2008_2010_s": t_filt_pq,
        })

    # -------------------------------------------------------------------------
    # Layout 5: Quarter-File Layout (REPDTE=YYYYMMDD.parquet, Snappy)
    # -------------------------------------------------------------------------
    print("4. Benchmarking Quarter-Sharded Parquet Layout (Snappy)...")
    size_part = dir_size_mb(local_part_dir)

    # Measure writing sharded files by REPDTE
    temp_part_write = work_dir / "temp_part_write"
    def write_sharded():
        if temp_part_write.exists():
            shutil.rmtree(temp_part_write)
        temp_part_write.mkdir(parents=True)
        for repdte, group_df in df_pandas.groupby("REPDTE", sort=False):
            group_df.to_parquet(temp_part_write / f"REPDTE={repdte}.parquet", index=False, compression="snappy")

    t_write_part = time_op(write_sharded, repeats=1)
    if temp_part_write.exists():
        shutil.rmtree(temp_part_write)

    t_full_part = time_op(lambda: pl.read_parquet(str(local_part_dir / "*.parquet")))
    t_proj_part = time_op(lambda: pl.read_parquet(str(local_part_dir / "*.parquet"), columns=PROJ_COLS))

    # Selective file pruning: only read files matching the target quarters
    def read_pruned_files():
        target_files = [
            p for p in local_part_dir.glob("REPDTE=*.parquet")
            if FILTER_START <= p.stem.split("=")[1] <= FILTER_END
        ]
        return pl.read_parquet(target_files, columns=PROJ_COLS)

    t_filt_part = time_op(read_pruned_files)

    results.append({
        "Layout": "Quarter-Sharded Parquet (REPDTE=q.parquet)",
        "Compression": "SNAPPY",
        "Size_MB": round(size_part, 2),
        "Write_s": t_write_part,
        "Read_Full_s": t_full_part,
        "Read_Proj_5Cols_s": t_proj_part,
        "Read_Filter_2008_2010_s": t_filt_part,
    })

    res_df = pd.DataFrame(results)
    res_df["Compression_Ratio_vs_CSV"] = (size_csv / res_df["Size_MB"]).round(2)
    res_df["Speedup_Proj_vs_CSV"] = (t_proj_csv / res_df["Read_Proj_5Cols_s"]).round(2)
    res_df["Speedup_Filter_vs_CSV"] = (t_filt_csv / res_df["Read_Filter_2008_2010_s"]).round(2)

    out_csv = out_dir / "part1_storage_benchmark.csv"
    res_df.to_csv(out_csv, index=False)
    print("\n=== STORAGE BENCHMARK RESULTS ===")
    print(res_df.to_string(index=False))
    return res_df


if __name__ == "__main__":
    data_dir = Path(os.environ.get("FDIC_DATA_DIR", "data_raw/fdic"))
    out_dir = Path(os.environ.get("FDIC_OUTPUT_DIR", "outputs"))
    local_work = Path("/content/benchmark_workspace")
    run_storage_benchmark(data_dir / "financials", local_work, out_dir)

1. Loading partitioned dataset into memory to prepare benchmark baselines...
   Loaded 1,682,616 rows x 33 columns.
2. Benchmarking Raw CSV...
3. Benchmarking Single Parquet (NONE)...
3. Benchmarking Single Parquet (SNAPPY)...
3. Benchmarking Single Parquet (ZSTD)...
4. Benchmarking Quarter-Sharded Parquet Layout (Snappy)...

=== STORAGE BENCHMARK RESULTS ===
                                    Layout Compression  Size_MB  Write_s  Read_Full_s  Read_Proj_5Cols_s  Read_Filter_2008_2010_s  Compression_Ratio_vs_CSV  Speedup_Proj_vs_CSV  Speedup_Filter_vs_CSV
                     Raw CSV (Single File)        None   422.52  96.8035       3.7614             0.8532                   1.1183                      1.00                 1.00                   1.00
                            Single Parquet        NONE   204.16   2.9653       0.2182             0.0251                   0.0046                      2.07                33.99                 243.11
                            Single Par

## Part 2: Data Quality Audit & Point-in-Time Panel Construction
**Objective:** Systematically detect regime-induced missingness and accounting anomalies, resolve them without look-ahead bias, and construct the target failure label ($y_{i,q}$).

**Sections:**
1. **Automated Data Quality Audit (`src/p2_data_quality.py`):** Scans the Bronze layer to map missing data to historical regulatory regimes (e.g., confidential pre-2001 delinquency schedules) and identifies structural anomalies such as the ROE sign-paradox when book equity is negative (`EQ <= 0`).
2. **Point-in-Time Panel Construction (`src/p2_build_panel.py`):**
   * Resolves bank exit taxonomy by cross-referencing failures and static directory tables.
   * Filters out post-failure receivership filings (`REPDTE >= FAILDATE`).
   * Constructs the 4-quarter forward failure label: $y_{i,q} = \mathbf{1}\{\text{bank } i \text{ fails in } (q, q+4\text{ quarters}]\}$.
   * Evaluates split candidates, locking the optimal chronological boundary at `2008Q4` with a 4-quarter purge gap.

**Pipeline Linkage:** Consolidates the audited data into the **Silver Panel** (`panel_silver.parquet`), filtering out unneeded metadata and passing a clean, contiguous longitudinal panel to Part 3 and Part 4.

In [6]:
"""
p2_data_quality.py
Automated Data Quality Audit for FDIC Call Reports (1984Q1 - 2026Q2), Failures, and Institutions.
Identifies regime-based missingness, accounting anomalies, ratio distortions,
temporal reporting gaps, and relational integrity issues across tables.
"""
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

FIN_NUMERIC_COLS = [
    "ASSET", "DEP", "DEPDOM", "EQ", "BRO", "CHBAL", "SC",
    "LNLSGR", "LNLSNET", "LNRE", "LNCI", "LNCON", "LNATRES",
    "NCLNLS", "P3ASSET", "P9ASSET", "NAASSET", "NTLNLS", "ORE",
    "NETINC", "INTINC", "EINTEXP", "NONII", "NONIX", "ELNATR",
    "ROA", "ROE", "NIMY",
    "RBC1AAJ", "RBCRWAJ",
]


def assign_regulatory_era(repdte_series: pd.Series) -> pd.Series:
    year = repdte_series.str.slice(0, 4).astype(int)
    return pd.Series(
        np.select(
            [
                year <= 1989,
                (year >= 1990) & (year <= 2007),
                (year >= 2008) & (year <= 2019),
                year >= 2020,
            ],
            [
                "1_Pre_Basel_I (1984-1989)",
                "2_Basel_I_Dereg (1990-2007)",
                "3_GFC_Basel_III (2008-2019)",
                "4_COVID_CBLR (2020-2026)",
            ],
            default="Unknown",
        ),
        index=repdte_series.index,
    )


def run_quality_audit(data_dir: Path, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    print("Loading datasets from Bronze storage...")
    fin = pq.read_table(data_dir / "financials").to_pandas()
    fails = pd.read_parquet(data_dir / "failures.parquet")
    insts = pd.read_parquet(data_dir / "institutions.parquet")

    fin = fin.sort_values(["CERT", "REPDTE"]).reset_index(drop=True)
    n_total = len(fin)
    print(f"Loaded Financials: {n_total:,} rows | Failures: {len(fails):,} | Institutions: {len(insts):,}\n")

    # =========================================================================
    # AUDIT 1: Missingness by Regulatory Era
    # =========================================================================
    fin["Era"] = assign_regulatory_era(fin["REPDTE"])
    era_groups = fin.groupby("Era", observed=True)

    miss_records = []
    for col in FIN_NUMERIC_COLS:
        overall_miss_pct = fin[col].isna().mean() * 100
        era_miss = era_groups[col].apply(lambda s: s.isna().mean() * 100).to_dict()
        miss_records.append({
            "Field": col,
            "Overall_Missing_%": round(overall_miss_pct, 2),
            "1984_1989_%": round(era_miss.get("1_Pre_Basel_I (1984-1989)", 0.0), 2),
            "1990_2007_%": round(era_miss.get("2_Basel_I_Dereg (1990-2007)", 0.0), 2),
            "2008_2019_%": round(era_miss.get("3_GFC_Basel_III (2008-2019)", 0.0), 2),
            "2020_2026_%": round(era_miss.get("4_COVID_CBLR (2020-2026)", 0.0), 2),
        })

    miss_df = pd.DataFrame(miss_records).sort_values("Overall_Missing_%", ascending=False)
    miss_df.to_csv(out_dir / "p2_audit_missingness_by_era.csv", index=False)

    # =========================================================================
    # AUDIT 2: Accounting Anomalies, Balance Sheet Identities & Ratio Distortions
    # =========================================================================
    anomalies = [
        ("Non-positive Total Assets (ASSET <= 0)", (fin["ASSET"] <= 0)),
        ("Negative Total Deposits (DEP < 0)", (fin["DEP"] < 0)),
        ("Technical Insolvency: Negative Equity (EQ < 0)", (fin["EQ"] < 0)),
        ("Zero Equity Boundary (EQ == 0)", (fin["EQ"] == 0)),
        ("Domestic Deposits Exceed Total Deposits (DEPDOM > DEP)", (fin["DEPDOM"] > fin["DEP"] + 1.0)),
        ("Net Loans Exceed Gross Loans (LNLSNET > LNLSGR)", (fin["LNLSNET"] > fin["LNLSGR"] + 1.0)),
        ("Loan Subcategories Exceed Gross Loans (LNRE+LNCI+LNCON > LNLSGR*1.05)",
         ((fin["LNRE"].fillna(0) + fin["LNCI"].fillna(0) + fin["LNCON"].fillna(0)) > fin["LNLSGR"] * 1.05) & (fin["LNLSGR"] > 0)),
        ("Negative Loan Loss Reserves (LNATRES < 0)", (fin["LNATRES"] < 0)),
        ("Negative Cash & Balances Due (CHBAL < 0)", (fin["CHBAL"] < 0)),
        ("Negative Investment Securities (SC < 0)", (fin["SC"] < 0)),
        ("Negative Brokered Deposits (BRO < 0)", (fin["BRO"] < 0)),
        ("Extreme ROE Distortion (|ROE| > 500%)", (fin["ROE"].abs() > 500)),
        ("Extreme ROA Distortion (|ROA| > 50%)", (fin["ROA"].abs() > 50)),
        ("Negative Equity & Negative Net Income Sign Paradox (EQ < 0 & NETINC < 0)",
         (fin["EQ"] < 0) & (fin["NETINC"] < 0)),
        ("Implausible Tier 1 Leverage Ratio (RBC1AAJ < -50% or > 100%)",
         (fin["RBC1AAJ"] < -50) | (fin["RBC1AAJ"] > 100)),
    ]

    anom_records = []
    for desc, mask in anomalies:
        cnt = int(mask.sum())
        anom_records.append({
            "Anomaly_Check": desc,
            "Row_Count": cnt,
            "Pct_of_Panel_%": round(cnt / n_total * 100, 4),
        })
    anom_df = pd.DataFrame(anom_records)
    anom_df.to_csv(out_dir / "p2_audit_accounting_anomalies.csv", index=False)

    # =========================================================================
    # AUDIT 3: Temporal Continuity & Relational Integrity (Failures & Institutions)
    # =========================================================================
    # Compute integer quarter index: year * 4 + (month // 3)
    yr = fin["REPDTE"].str.slice(0, 4).astype(int)
    mo = fin["REPDTE"].str.slice(4, 6).astype(int)
    fin["q_idx"] = yr * 4 + (mo // 3)
    fin["q_diff"] = fin.groupby("CERT")["q_idx"].diff()

    n_gaps = int((fin["q_diff"] > 1).sum())
    n_banks_with_gaps = int(fin.loc[fin["q_diff"] > 1, "CERT"].nunique())
    max_gap_quarters = int(fin["q_diff"].max()) if n_gaps > 0 else 0

    # Relational checks across tables
    fin_certs = set(fin["CERT"].dropna().unique())
    inst_certs = set(insts["CERT"].dropna().unique())

    fails_null_cert = int(fails["CERT"].isna().sum())
    fails_valid = fails.dropna(subset=["CERT"]).copy()
    fails_valid["CERT"] = fails_valid["CERT"].astype(int)
    fails_Valid_certs = set(fails_valid["CERT"].unique())
    fails_dup_certs = int(fails_valid.duplicated(subset=["CERT"]).sum())

    # Parse failure dates
    fails_valid["FAILDATE_DT"] = pd.to_datetime(fails_valid["FAILDATE"], format="mixed", errors="coerce")
    fails_post_1984 = fails_valid[fails_valid["FAILDATE_DT"] >= "1984-01-01"]
    fails_in_fin = len(set(fails_post_1984["CERT"]).intersection(fin_certs))
    fails_not_in_fin = len(set(fails_post_1984["CERT"]) - fin_certs)

    # Check post-failure Call Report filings (REPDTE >= FAILDATE)
    first_fail = fails_valid.groupby("CERT")["FAILDATE_DT"].min().rename("FIRST_FAILDATE")
    last_fail = fails_valid.groupby("CERT")["FAILDATE_DT"].max().rename("LAST_FAILDATE")
    fin_fail_check = fin[["CERT", "REPDTE"]].merge(first_fail, on="CERT", how="inner").merge(last_fail, on="CERT", how="inner")
    fin_fail_check["REPDTE_DT"] = pd.to_datetime(fin_fail_check["REPDTE"], format="%Y%m%d")

    post_first_fail_rows = int((fin_fail_check["REPDTE_DT"] >= fin_fail_check["FIRST_FAILDATE"]).sum())
    post_last_fail_rows = int((fin_fail_check["REPDTE_DT"] >= fin_fail_check["LAST_FAILDATE"]).sum())

    integrity_records = [
        {"Metric": "Total Bank-Quarter Observations", "Count": n_total},
        {"Metric": "Distinct Banks in Financials (CERT)", "Count": len(fin_certs)},
        {"Metric": "Banks in Financials Missing from Institutions Table", "Count": len(fin_certs - inst_certs)},
        {"Metric": "Internal Reporting Gaps (q_diff > 1 quarter)", "Count": n_gaps},
        {"Metric": "Distinct Banks with Reporting Gaps", "Count": n_banks_with_gaps},
        {"Metric": "Maximum Reporting Gap Length (Quarters)", "Count": max_gap_quarters},
        {"Metric": "Failure Records with Missing CERT (Unlinkable Thrifts)", "Count": fails_null_cert},
        {"Metric": "Duplicate CERT Records in Failures Table (Repeat/Assisted)", "Count": fails_dup_certs},
        {"Metric": "Post-1984 Failed CERTs Matched in Financials Panel", "Count": fails_in_fin},
        {"Metric": "Post-1984 Failed CERTs Without Call Reports in Panel", "Count": fails_not_in_fin},
        {"Metric": "Call Reports Filed ON or AFTER First Failure Date", "Count": post_first_fail_rows},
        {"Metric": "Call Reports Filed ON or AFTER Last Failure Date", "Count": post_last_fail_rows},
    ]
    integ_df = pd.DataFrame(integrity_records)
    integ_df.to_csv(out_dir / "p2_audit_temporal_and_relational.csv", index=False)

    print("=== 1. MISSINGNESS BY REGULATORY ERA (TOP 15 FIELDS) ===")
    print(miss_df.head(15).to_string(index=False))
    print("\n=== 2. ACCOUNTING & RATIO ANOMALIES ===")
    print(anom_df.to_string(index=False))
    print("\n=== 3. TEMPORAL CONTINUITY & RELATIONAL INTEGRITY ===")
    print(integ_df.to_string(index=False))


if __name__ == "__main__":
    data_dir = Path(os.environ.get("FDIC_DATA_DIR", "data_raw/fdic"))
    out_dir = Path(os.environ.get("FDIC_OUTPUT_DIR", "outputs"))
    run_quality_audit(data_dir, out_dir)

Loading datasets from Bronze storage...
Loaded Financials: 1,682,616 rows | Failures: 4,117 | Institutions: 27,835

=== 1. MISSINGNESS BY REGULATORY ERA (TOP 15 FIELDS) ===
  Field  Overall_Missing_%  1984_1989_%  1990_2007_%  2008_2019_%  2020_2026_%
P3ASSET              57.76       100.00        68.19         0.00         0.00
 NTLNLS               4.75        18.25         0.26         0.25         0.36
NAASSET               4.62        18.53         0.00         0.00         0.00
P9ASSET               4.62        18.53         0.00         0.00         0.00
 ELNATR               2.57         9.49         0.26         0.25         0.36
    ROE               1.11         2.97         0.60         0.26         0.47
    BRO               1.00         3.19         0.26         0.25         0.36
EINTEXP               0.38         0.73         0.26         0.25         0.36
 NETINC               0.28         0.32         0.26         0.25         0.36
  NONIX               0.28         0.

In [7]:
"""
p2_build_panel.py
Constructs the point-in-time clean bank-quarter Silver panel (1984Q1 - 2026Q2),
resolves bank exits (Failures vs. Mergers/Other Exits vs. Active), computes the
4-quarter forward failure label y_{i,q}, and saves the consolidated dataset.
"""
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


def build_quarter_lookup(unique_repdtes: list[str]) -> pd.DataFrame:
    """Build exact calendar lookup for quarter index and +4 quarter end dates."""
    rows = []
    for q_str in sorted(unique_repdtes):
        y = int(q_str[:4])
        m = int(q_str[4:6])
        d = int(q_str[6:8])
        q_idx = y * 4 + (m // 3)
        repdte_dt = pd.Timestamp(year=y, month=m, day=d)
        # Exactly 4 quarters later is (y + 1, m, d) since quarter-ends are Mar 31, Jun 30, Sep 30, Dec 31
        plus4q_dt = pd.Timestamp(year=y + 1, month=m, day=d)
        rows.append({
            "REPDTE": q_str,
            "q_idx": q_idx,
            "REPDTE_DT": repdte_dt,
            "PLUS4Q_DT": plus4q_dt,
        })
    return pd.DataFrame(rows)


def build_clean_panel(data_dir: Path, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    silver_dir = data_dir.parent / "data_processed"
    silver_dir.mkdir(parents=True, exist_ok=True)

    print("1. Loading raw Bronze tables...")
    fin = pq.read_table(data_dir / "financials").to_pandas()
    fails = pd.read_parquet(data_dir / "failures.parquet")
    insts = pd.read_parquet(data_dir / "institutions.parquet")

    # Sort deterministically by (CERT, REPDTE)
    fin = fin.sort_values(["CERT", "REPDTE"]).reset_index(drop=True)
    n_raw = len(fin)

    # -------------------------------------------------------------------------
    # 2. Point-in-Time Data Cleaning (Zero Look-Ahead Bias)
    # -------------------------------------------------------------------------
    print("2. Applying point-in-time accounting and domain cleaning rules...")
    # Rule A: Filter impossible non-positive assets or negative deposits
    valid_scale_mask = (fin["ASSET"] > 0) & (fin["DEP"].fillna(0) >= 0)
    n_dropped_scale = int((~valid_scale_mask).sum())
    fin = fin.loc[valid_scale_mask].copy().reset_index(drop=True)

    # Rule B: Preserve negative equity (technical insolvency) via explicit indicator
    fin["flag_neg_equity"] = (fin["EQ"] <= 0).astype("int8")

    # Rule C: When EQ <= 0, raw ROE (NETINC / EQ) suffers from denominator explosion or sign inversion
    # Flag unreliable ROE observations without peering into future distributions
    fin["flag_roe_Signing_distortion"] = ((fin["EQ"] <= 0) | (fin["ROE"].abs() > 500)).astype("int8")

    # Rule D: Fix sub-component boundary violations point-in-time
    # Domestic deposits cannot exceed total deposits
    depdom_exceed = fin["DEPDOM"] > fin["DEP"]
    fin.loc[depdom_exceed, "DEPDOM"] = fin.loc[depdom_exceed, "DEP"]

    # Net loans cannot exceed gross loans
    lnnet_exceed = fin["LNLSNET"] > fin["LNLSGR"]
    fin.loc[lnnet_exceed, "LNLSNET"] = fin.loc[lnnet_exceed, "LNLSGR"]

    # Strictly non-negative balance sheet stock items: replace negative reporting errors with NaN
    for non_neg_col in ["LNATRES", "CHBAL", "SC", "BRO", "ORE", "NCLNLS", "P3ASSET", "P9ASSET", "NAASSET"]:
        neg_mask = fin[non_neg_col] < 0
        if neg_mask.any():
            fin.loc[neg_mask, non_neg_col] = np.nan

    # -------------------------------------------------------------------------
    # 3. Join Point-in-Time Static Institution Attributes (No Future Leakage)
    # -------------------------------------------------------------------------
    print("3. Joining static charter attributes and auditing temporal continuity...")
    # CRITICAL: Only BKCLASS, ESTYMD, INSDATE are known at charter inception.
    # ACTIVE, ENDEFYMD, CHANGEC1 are end-of-sample (2026) retrospective fields and
    # are kept strictly for exit taxonomy auditing, NEVER for feature engineering.
    inst_sub = insts[["CERT", "BKCLASS", "ESTYMD", "INSDATE", "ACTIVE", "ENDEFYMD", "CHANGEC1"]].drop_duplicates("CERT")
    fin = fin.merge(inst_sub, on="CERT", how="left")

    # Attach exact calendar quarter index
    q_lookup = build_quarter_lookup(fin["REPDTE"].unique().tolist())
    fin = fin.merge(q_lookup, on="REPDTE", how="left")
    fin = fin.sort_values(["CERT", "q_idx"]).reset_index(drop=True)

    # Calculate quarter gap from previous report of the same bank
    fin["q_gap"] = fin.groupby("CERT")["q_idx"].diff().fillna(1).astype("int16")
    fin["has_reporting_gap"] = (fin["q_gap"] > 1).astype("int8")

    # Causal 1-quarter forward fill ONLY for contiguous quarters (q_gap == 1) on core balance sheet stocks
    # Never fill across reporting gaps (q_gap > 1) and never backward-fill!
    core_stocks = ["ASSET", "DEP", "DEPDOM", "EQ", "LNLSGR", "LNLSNET", "LNATRES", "CHBAL", "SC"]
    for col in core_stocks:
        prev_val = fin.groupby("CERT")[col].shift(1)
        fill_mask = fin[col].isna() & (fin["q_gap"] == 1) & prev_val.notna()
        fin.loc[fill_mask, col] = prev_val.loc[fill_mask]

    # -------------------------------------------------------------------------
    # 4. Point-in-Time Failure Matching & Label Construction y_{i,q}
    # -------------------------------------------------------------------------
    print("4. Matching failure events and constructing 4-quarter forward label y_{i,q}...")
    fails_clean = fails.dropna(subset=["CERT"]).copy()
    fails_clean["CERT"] = fails_clean["CERT"].astype("int64")
    fails_clean["FAILDATE_DT"] = pd.to_datetime(fails_clean["FAILDATE"], format="mixed", errors="coerce")
    fails_clean = fails_clean.dropna(subset=["FAILDATE_DT"]).sort_values(["CERT", "FAILDATE_DT"])

    # Determine each failed bank's terminal (last) failure date and earliest failure date
    fail_summary = fails_clean.groupby("CERT").agg(
        first_fail_dt=("FAILDATE_DT", "min"),
        last_fail_dt=("FAILDATE_DT", "max"),
        fail_count=("FAILDATE_DT", "count"),
        fail_cost=("COST", "sum"),
        last_restype=("RESTYPE", "last"),
    ).reset_index()

    fin = fin.merge(fail_summary, on="CERT", how="left")

    # For banks with failure records, find the next failure date strictly after REPDTE_DT
    # Because only ~4,100 banks ever fail, we compute next_fail_dt on the subset of failed CERTs
    failed_certs_panel = fin.loc[fin["first_fail_dt"].notna(), ["CERT", "REPDTE", "REPDTE_DT"]].copy()
    merged_events = failed_certs_panel.merge(
        fails_clean[["CERT", "FAILDATE_DT", "COST", "RESTYPE"]],
        on="CERT",
        how="inner",
    )
    # Strictly future failures relative to quarter-end REPDTE_DT: FAILDATE > REPDTE
    future_events = merged_events[merged_events["FAILDATE_DT"] > merged_events["REPDTE_DT"]].copy()
    next_event = (
        future_events.sort_values(["CERT", "REPDTE", "FAILDATE_DT"])
        .groupby(["CERT", "REPDTE"], as_index=False)
        .first()[["CERT", "REPDTE", "FAILDATE_DT", "COST", "RESTYPE"]]
        .rename(columns={
            "FAILDATE_DT": "next_fail_dt",
            "COST": "next_fail_cost",
            "RESTYPE": "next_fail_restype",
        })
    )

    fin = fin.merge(next_event, on=["CERT", "REPDTE"], how="left")

    # Flag post-failure receivership filings (REPDTE_DT >= last_fail_dt)
    # Also flag any filing on/after first_fail_dt unless the first event was Open Bank Assistance ('OB')
    fin["is_post_failure"] = (
        (fin["last_fail_dt"].notna() & (fin["REPDTE_DT"] >= fin["last_fail_dt"]))
        | (
            fin["first_fail_dt"].notna()
            & (fin["REPDTE_DT"] >= fin["first_fail_dt"])
            & (fin["next_fail_dt"].isna())
        )
    ).astype("int8")

    # Target Label Definition: y_{i,q} = 1{ bank i fails in (q, q + 4 quarters] }
    fin["target_fail_4q"] = (
        (fin["is_post_failure"] == 0)
        & fin["next_fail_dt"].notna()
        & (fin["next_fail_dt"] <= fin["PLUS4Q_DT"])
    ).astype("int8")

    # Modeling scope flag: up to 2025Q2 (20250630) and excluding post-failure rows
    fin["in_modeling_sample"] = (
        (fin["REPDTE"] <= "20250630") & (fin["is_post_failure"] == 0)
    ).astype("int8")

    # -------------------------------------------------------------------------
    # 5. Bank-Level Exit Taxonomy (Failure vs. Merger/Other Exit vs. Active)
    # -------------------------------------------------------------------------
    print("5. Classifying bank exits (Failures vs. Mergers/Other Exits vs. Active)...")
    active_panel = fin[fin["is_post_failure"] == 0]
    bank_span = active_panel.groupby("CERT").agg(
        first_repdte=("REPDTE", "min"),
        last_repdte=("REPDTE", "max"),
        last_repdte_dt=("REPDTE_DT", "max"),
        last_plus4q_dt=("PLUS4Q_DT", "max"),
        n_quarters=("REPDTE", "count"),
        ever_y1=("target_fail_4q", "max"),
        has_fail_record=("first_fail_dt", lambda s: s.notna().any()),
        last_fail_dt=("last_fail_dt", "max"),
        inst_active=("ACTIVE", "last"),
        inst_changec1=("CHANGEC1", "last"),
    ).reset_index()

    # Classify each distinct bank in the panel:
    # 1. Failed within 4 quarters of last Call Report
    # 2. Failed (>4 quarters after last Call Report or Assisted Open-Bank resolution)
    # 3. Active as of 2026Q2 (last_repdte == '20260630')
    # 4. Merged / Consolidated / Voluntary Exit (stopped reporting before 2026Q2 without failure)
    cond_fail_4q = bank_span["ever_y1"] == 1
    cond_fail_other = (~cond_fail_4q) & bank_span["has_fail_record"]
    cond_active = (~cond_fail_4q) & (~cond_fail_other) & (bank_span["last_repdte"] == "20260630")

    bank_span["exit_category"] = np.select(
        [cond_fail_4q, cond_fail_other, cond_active],
        [
            "1_Failed (Captured in 4Q Horizon)",
            "2_Failed (Assisted / Delayed >4Q Resolution)",
            "3_Active (Operating at 2026Q2)",
        ],
        default="4_Merged_or_Voluntary_Exit (Non-Failure)",
    )

    exit_summary = bank_span.groupby("exit_category").agg(
        Distinct_Banks=("CERT", "count"),
        Total_Bank_Quarters=("n_quarters", "sum"),
        Positive_Labels_y1=("ever_y1", "sum"),
    ).reset_index()
    exit_summary["Pct_of_Banks_%"] = (exit_summary["Distinct_Banks"] / len(bank_span) * 100).round(2)
    exit_summary.to_csv(out_dir / "p2_panel_exit_taxonomy.csv", index=False)

    # Merge exit_category back to panel for reference (audit-only metadata)
    fin = fin.merge(bank_span[["CERT", "exit_category"]], on="CERT", how="left")

    # -------------------------------------------------------------------------
    # 6. Verify Development vs. Test Period Failure Distribution (Up to 2025Q2)
    # -------------------------------------------------------------------------
    mod_df = fin[fin["in_modeling_sample"] == 1].copy()
    mod_df["Year"] = mod_df["REPDTE"].str.slice(0, 4).astype(int)

    # Evaluate candidate chronological splits with a 4-quarter purge gap:
    # Candidate Split A: Dev 1984Q1-2008Q4 | Gap 2009Q1-2009Q4 (4Q) | Test 2010Q1-2025Q2
    # Candidate Split B: Dev 1984Q1-2009Q2 | Gap 2009Q3-2010Q2 (4Q) | Test 2010Q3-2025Q2
    # Candidate Split C: Dev 1984Q1-2009Q4 | Gap 2010Q1-2010Q4 (4Q) | Test 2011Q1-2025Q2
    split_candidates = [
        ("Split_A (Dev<=2008Q4, Gap=2009, Test=2010Q1-2025Q2)", "20081231", "20100331"),
        ("Split_B (Dev<=2009Q2, Gap=2009Q3-2010Q2, Test=2010Q3-2025Q2)", "20090630", "20100930"),
        ("Split_C (Dev<=2009Q4, Gap=2010, Test=2011Q1-2025Q2)", "20091231", "20110331"),
    ]
    split_rows = []
    for name, dev_end, test_start in split_candidates:
        dev_sub = mod_df[mod_df["REPDTE"] <= dev_end]
        test_sub = mod_df[mod_df["REPDTE"] >= test_start]
        dev_failed_banks = int(dev_sub.loc[dev_sub["target_fail_4q"] == 1, "CERT"].nunique())
        test_failed_banks = int(test_sub.loc[test_sub["target_fail_4q"] == 1, "CERT"].nunique())
        split_rows.append({
            "Candidate_Split": name,
            "Dev_Rows": len(dev_sub),
            "Dev_Pos_Labels_y1": int(dev_sub["target_fail_4q"].sum()),
            "Dev_Failed_Banks": dev_failed_banks,
            "Test_Rows": len(test_sub),
            "Test_Pos_Labels_y1": int(test_sub["target_fail_4q"].sum()),
            "Test_Failed_Banks": test_failed_banks,
            "Meets_Min_50_Failures": test_failed_banks >= 50,
        })
    split_df = pd.DataFrame(split_rows)
    split_df.to_csv(out_dir / "p2_split_candidates_summary.csv", index=False)

    # Save Silver Panel as consolidated Snappy-compressed Parquet
    # Drop temporary datetime objects that duplicate string/integer keys or convert cleanly
    silver_path = silver_dir / "panel_silver.parquet"
    fin.to_parquet(silver_path, index=False, compression="snappy")
    silver_size_mb = silver_path.stat().st_size / (1024 * 1024)

    print("\n=== PANEL RECONCILIATION SUMMARY ===")
    print(f"Raw Bronze Rows                 : {n_raw:,}")
    print(f"Dropped (ASSET<=0 or DEP<0)     : {n_dropped_scale:,}")
    print(f"Clean Silver Panel Rows         : {len(fin):,}")
    print(f"Post-Failure Rows Flagged       : {int(fin['is_post_failure'].sum()):,}")
    print(f"Modeling Sample Rows (<=2025Q2) : {len(mod_df):,}")
    print(f"Total Positive Labels (y=1)     : {int(mod_df['target_fail_4q'].sum()):,}")
    print(f"Distinct Failed Banks (y=1)     : {int(mod_df.loc[mod_df['target_fail_4q']==1, 'CERT'].nunique()):,}")
    print(f"Silver Parquet Size (Snappy)    : {silver_size_mb:.2f} MB -> {silver_path}")

    print("\n=== BANK EXIT TAXONOMY (DISTINCT BANKS & QUARTERS) ===")
    print(exit_summary.to_string(index=False))

    print("\n=== CANDIDATE DEVELOPMENT / TEST SPLITS (WITH 4-QUARTER PURGE GAP) ===")
    print(split_df.to_string(index=False))


if __name__ == "__main__":
    data_dir = Path(os.environ.get("FDIC_DATA_DIR", "data_raw/fdic"))
    out_dir = Path(os.environ.get("FDIC_OUTPUT_DIR", "outputs"))
    build_clean_panel(data_dir, out_dir)

1. Loading raw Bronze tables...
2. Applying point-in-time accounting and domain cleaning rules...
3. Joining static charter attributes and auditing temporal continuity...
4. Matching failure events and constructing 4-quarter forward label y_{i,q}...
5. Classifying bank exits (Failures vs. Mergers/Other Exits vs. Active)...

=== PANEL RECONCILIATION SUMMARY ===
Raw Bronze Rows                 : 1,682,616
Dropped (ASSET<=0 or DEP<0)     : 0
Clean Silver Panel Rows         : 1,682,616
Post-Failure Rows Flagged       : 4,799
Modeling Sample Rows (<=2025Q2) : 1,660,316
Total Positive Labels (y=1)     : 12,459
Distinct Failed Banks (y=1)     : 3,216
Silver Parquet Size (Snappy)    : 166.41 MB -> /content/drive/MyDrive/BFT65B_Project/data_raw/data_processed/panel_silver.parquet

=== BANK EXIT TAXONOMY (DISTINCT BANKS & QUARTERS) ===
                           exit_category  Distinct_Banks  Total_Bank_Quarters  Positive_Labels_y1  Pct_of_Banks_%
       1_Failed (Captured in 4Q Horizon)        

## Part 3: Scalable Data Processing (Polars, Pandas & PySpark)
**Objective:** Implement the core analytical transformations (CAMELS ratios, trailing windows, and cross-sectional aggregates) and benchmark execution engines to determine when a distributed Spark cluster is justified over a local laptop.

**Sections:**

###3.1. **Polars vs. Pandas Engine Benchmark (`src/p3_polars_vs_pandas_benchmark.py`):**
   * Implements 4 transformation stages: I/O Filter $\rightarrow$ Vectorized Ratios $\rightarrow$ Longitudinal Windows (`over("CERT")`) $\rightarrow$ Cross-Sectional Windows (`over("REPDTE")`).
   * Captures the Polars Lazy query optimizer plan (Projection Pushdown & Common Subexpression Elimination).
   * Benchmarks Stage-by-Stage runtime on the $1.0\times$ panel and multi-scale execution from $0.25\times$ to $5.00\times$ (8.4 million rows) against an eager Pandas baseline.

**Pipeline Linkage:** Proves that the computational logic is sound and highly optimized before expanding the full 51-column feature matrix in Part 4.

In [8]:
"""
p3_polars_vs_pandas_benchmark.py
Implements core panel transformations in Polars (Lazy & Eager) and Pandas.
Captures Polars query optimization plans, benchmarks stage-by-stage latency,
and evaluates memory/runtime scaling from 0.25x (420k rows) to 5.0x (8.4M rows).
"""
from __future__ import annotations

import gc
import os
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import psutil
import pyarrow.parquet as pq

REQUIRED_COLS = [
    "CERT", "REPDTE", "STNAME", "q_idx", "is_post_failure",
    "ASSET", "DEP", "EQ", "BRO", "CHBAL", "SC",
    "LNLSGR", "LNRE", "LNCI", "LNATRES", "NCLNLS",
    "INTINC", "NONII", "NONIX", "ROA", "NIMY", "RBC1AAJ",
]

 FLOAT_COLS = [
    "ASSET", "DEP", "EQ", "BRO", "CHBAL", "SC",
    "LNLSGR", "LNRE", "LNCI", "LNATRES", "NCLNLS",
    "INTINC", "NONII", "NONIX", "ROA", "NIMY", "RBC1AAJ",
]


def get_rss_mb() -> float:
    return psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)


def locate_silver_parquet(data_dir: Path) -> Path:
    candidates = [
        data_dir.parent / "data_processed" / "panel_silver.parquet",
        data_dir / "data_processed" / "panel_silver.parquet",
        Path("/content/drive/MyDrive/BFT65B_Project/data_raw/data_processed/panel_silver.parquet"),
        Path("/content/drive/MyDrive/BFT65B_Project/data_processed/panel_silver.parquet"),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Cannot locate panel_silver.parquet in {candidates}")


# =============================================================================
# 1. POLARS LAZY PIPELINE (Idiomatic Query-Optimized Implementation)
# =============================================================================
def build_polars_lazy_query(parquet_path: Path) -> pl.LazyFrame:
    lf = (
        pl.scan_parquet(parquet_path)
        .filter(pl.col("is_post_failure") == 0)
        .select([c for c in REQUIRED_COLS if c != "is_post_failure"])
        .with_columns([pl.col(c).fill_nan(None) for c in FLOAT_COLS])
        .sort(["CERT", "q_idx"])
    )

    # Stage 2: Vectorized CAMELS Ratios (%)
    lf = lf.with_columns([
        (pl.col("EQ") / pl.col("ASSET") * 100.0).alias("eq_to_asset"),
        pl.when(pl.col("LNLSGR") > 0)
          .then(pl.col("NCLNLS") / pl.col("LNLSGR") * 100.0)
          .otherwise(None).alias("npl_to_loans"),
        pl.when(pl.col("LNLSGR") > 0)
          .then(pl.col("LNATRES") / pl.col("LNLSGR") * 100.0)
          .otherwise(None).alias("reserves_to_loans"),
        pl.when(pl.col("NCLNLS") > 0)
          .then(pl.col("LNATRES") / pl.col("NCLNLS") * 100.0)
          .otherwise(None).alias("reserves_to_npl"),
        pl.when((pl.col("INTINC") + pl.col("NONII")) > 0)
          .then(pl.col("NONIX") / (pl.col("INTINC") + pl.col("NONII")) * 100.0)
          .otherwise(None).alias("cost_to_income"),
        pl.when(pl.col("DEP") > 0)
          .then(pl.col("BRO") / pl.col("DEP") * 100.0)
          .otherwise(None).alias("brokered_to_dep"),
        ((pl.col("CHBAL").fill_null(0.0) + pl.col("SC").fill_null(0.0)) / pl.col("ASSET") * 100.0).alias("liquid_to_asset"),
        pl.when(pl.col("DEP") > 0)
          .then(pl.col("LNLSGR") / pl.col("DEP") * 100.0)
          .otherwise(None).alias("loan_to_dep"),
        pl.when(pl.col("LNLSGR") > 0)
          .then(pl.col("LNRE") / pl.col("LNLSGR") * 100.0)
          .otherwise(None).alias("re_loan_share"),
        pl.when(pl.col("LNLSGR") > 0)
          .then(pl.col("LNCI") / pl.col("LNLSGR") * 100.0)
          .otherwise(None).alias("ci_loan_share"),
    ])

    # Stage 3: Longitudinal Window Operations over("CERT") with exact continuity guard
    is_contig_4q = (pl.col("q_idx") - pl.col("q_idx").shift(4).over("CERT")) == 4
    is_contig_roll4 = (pl.col("q_idx") - pl.col("q_idx").shift(3).over("CERT")) == 3

    lf = lf.with_columns([
        pl.when(is_contig_4q & (pl.col("ASSET").shift(4).over("CERT") > 0))
          .then((pl.col("ASSET") / pl.col("ASSET").shift(4).over("CERT") - 1.0) * 100.0)
          .otherwise(None).alias("asset_growth_yoy"),
        pl.when(is_contig_4q & (pl.col("DEP").shift(4).over("CERT") > 0))
          .then((pl.col("DEP") / pl.col("DEP").shift(4).over("CERT") - 1.0) * 100.0)
          .otherwise(None).alias("dep_growth_yoy"),
        pl.when(is_contig_4q)
          .then(pl.col("eq_to_asset") - pl.col("eq_to_asset").shift(4).over("CERT"))
          .otherwise(None).alias("eq_to_asset_chg_4q"),
        pl.when(is_contig_4q)
          .then(pl.col("npl_to_loans") - pl.col("npl_to_loans").shift(4).over("CERT"))
          .otherwise(None).alias("npl_ratio_chg_4q"),
        pl.when(is_contig_roll4)
          .then(pl.col("ROA").rolling_mean(window_size=4).over("CERT"))
          .otherwise(None).alias("roa_roll4_mean"),
        pl.when(is_contig_roll4 & pl.col("ROA").rolling_mean(window_size=4).over("CERT").is_not_null())
          .then(pl.col("ROA").rolling_std(window_size=4, ddof=1).over("CERT").fill_nan(0.0))
          .otherwise(None).alias("roa_roll4_std"),
        pl.when(is_contig_roll4)
          .then(pl.col("NIMY").rolling_mean(window_size=4).over("CERT"))
          .otherwise(None).alias("nim_roll4_mean"),
    ])

    # Stage 4: Cross-Sectional Window Operations over("REPDTE") & over(["REPDTE", "STNAME"])
    lf = lf.with_columns([
        ((pl.col("ROA") - pl.col("ROA").mean().over("REPDTE"))
         / pl.col("ROA").std(ddof=1).over("REPDTE")).alias("roa_peer_zscore"),
        ((pl.col("RBC1AAJ") - pl.col("RBC1AAJ").mean().over("REPDTE"))
         / pl.col("RBC1AAJ").std(ddof=1).over("REPDTE")).alias("tier1_peer_zscore"),
        pl.col("npl_to_loans").mean().over(["REPDTE", "STNAME"]).alias("state_npl_mean"),
        pl.col("ROA").median().over(["REPDTE", "STNAME"]).alias("state_roa_median"),
        pl.col("CERT").count().over(["REPDTE", "STNAME"]).cast(pl.Int32).alias("state_bank_count"),
    ])
    return lf


# =============================================================================
# 2. POLARS EAGER PIPELINE (No Lazy Query Optimizer / No Pushdown)
# =============================================================================
def run_polars_eager(parquet_path: Path, stage_times: dict | None = None) -> pl.DataFrame:
    t0 = time.perf_counter()
    # Eager read loads full table before filtering and projecting
    df = pl.read_parquet(parquet_path)
    df = df.filter(pl.col("is_post_failure") == 0).select([c for c in REQUIRED_COLS if c != "is_post_failure"])
    df = df.with_columns([pl.col(c).fill_nan(None) for c in FLOAT_COLS]).sort(["CERT", "q_idx"])
    if stage_times is not None:
        stage_times["Stage1_IO_Filter_s"] = round(time.perf_counter() - t0, 4)

    t1 = time.perf_counter()
    df = df.with_columns([
        (pl.col("EQ") / pl.col("ASSET") * 100.0).alias("eq_to_asset"),
        pl.when(pl.col("LNLSGR") > 0).then(pl.col("NCLNLS") / pl.col("LNLSGR") * 100.0).otherwise(None).alias("npl_to_loans"),
        pl.when(pl.col("LNLSGR") > 0).then(pl.col("LNATRES") / pl.col("LNLSGR") * 100.0).otherwise(None).alias("reserves_to_loans"),
        pl.when(pl.col("NCLNLS") > 0).then(pl.col("LNATRES") / pl.col("NCLNLS") * 100.0).otherwise(None).alias("reserves_to_npl"),
        pl.when((pl.col("INTINC") + pl.col("NONII")) > 0).then(pl.col("NONIX") / (pl.col("INTINC") + pl.col("NONII")) * 100.0).otherwise(None).alias("cost_to_income"),
        pl.when(pl.col("DEP") > 0).then(pl.col("BRO") / pl.col("DEP") * 100.0).otherwise(None).alias("brokered_to_dep"),
        ((pl.col("CHBAL").fill_null(0.0) + pl.col("SC").fill_null(0.0)) / pl.col("ASSET") * 100.0).alias("liquid_to_asset"),
        pl.when(pl.col("DEP") > 0).then(pl.col("LNLSGR") / pl.col("DEP") * 100.0).otherwise(None).alias("loan_to_dep"),
        pl.when(pl.col("LNLSGR") > 0).then(pl.col("LNRE") / pl.col("LNLSGR") * 100.0).otherwise(None).alias("re_loan_share"),
        pl.when(pl.col("LNLSGR") > 0).then(pl.col("LNCI") / pl.col("LNLSGR") * 100.0).otherwise(None).alias("ci_loan_share"),
    ])
    if stage_times is not None:
        stage_times["Stage2_Ratios_s"] = round(time.perf_counter() - t1, 4)

    t2 = time.perf_counter()
    is_contig_4q = (pl.col("q_idx") - pl.col("q_idx").shift(4).over("CERT")) == 4
    is_contig_roll4 = (pl.col("q_idx") - pl.col("q_idx").shift(3).over("CERT")) == 3
    df = df.with_columns([
        pl.when(is_contig_4q & (pl.col("ASSET").shift(4).over("CERT") > 0))
          .then((pl.col("ASSET") / pl.col("ASSET").shift(4).over("CERT") - 1.0) * 100.0).otherwise(None).alias("asset_growth_yoy"),
        pl.when(is_contig_4q & (pl.col("DEP").shift(4).over("CERT") > 0))
          .then((pl.col("DEP") / pl.col("DEP").shift(4).over("CERT") - 1.0) * 100.0).otherwise(None).alias("dep_growth_yoy"),
        pl.when(is_contig_4q).then(pl.col("eq_to_asset") - pl.col("eq_to_asset").shift(4).over("CERT")).otherwise(None).alias("eq_to_asset_chg_4q"),
        pl.when(is_contig_4q).then(pl.col("npl_to_loans") - pl.col("npl_to_loans").shift(4).over("CERT")).otherwise(None).alias("npl_ratio_chg_4q"),
        pl.when(is_contig_roll4).then(pl.col("ROA").rolling_mean(window_size=4).over("CERT")).otherwise(None).alias("roa_roll4_mean"),
        pl.when(is_contig_roll4 & pl.col("ROA").rolling_mean(window_size=4).over("CERT").is_not_null())
          .then(pl.col("ROA").rolling_std(window_size=4, ddof=1).over("CERT").fill_nan(0.0)).otherwise(None).alias("roa_roll4_std"),
        pl.when(is_contig_roll4).then(pl.col("NIMY").rolling_mean(window_size=4).over("CERT")).otherwise(None).alias("nim_roll4_mean"),
    ])
    if stage_times is not None:
        stage_times["Stage3_Longitudinal_Windows_s"] = round(time.perf_counter() - t2, 4)

    t3 = time.perf_counter()
    df = df.with_columns([
        ((pl.col("ROA") - pl.col("ROA").mean().over("REPDTE")) / pl.col("ROA").std(ddof=1).over("REPDTE")).alias("roa_peer_zscore"),
        ((pl.col("RBC1AAJ") - pl.col("RBC1AAJ").mean().over("REPDTE")) / pl.col("RBC1AAJ").std(ddof=1).over("REPDTE")).alias("tier1_peer_zscore"),
        pl.col("npl_to_loans").mean().over(["REPDTE", "STNAME"]).alias("state_npl_mean"),
        pl.col("ROA").median().over(["REPDTE", "STNAME"]).alias("state_roa_median"),
        pl.col("CERT").count().over(["REPDTE", "STNAME"]).cast(pl.Int32).alias("state_bank_count"),
    ])
    if stage_times is not None:
        stage_times["Stage4_CrossSection_Windows_s"] = round(time.perf_counter() - t3, 4)
    return df


# =============================================================================
# 3. PANDAS BASELINE PIPELINE (Vectorized GroupBy / Transform / Rolling)
# =============================================================================
def run_pandas_baseline(parquet_path: Path, stage_times: dict | None = None) -> pd.DataFrame:
    t0 = time.perf_counter()
    df = pd.read_parquet(parquet_path, columns=REQUIRED_COLS)
    df = df[df["is_post_failure"] == 0].drop(columns=["is_post_failure"])
    df = df.sort_values(["CERT", "q_idx"]).reset_index(drop=True)
    if stage_times is not None:
        stage_times["Stage1_IO_Filter_s"] = round(time.perf_counter() - t0, 4)

    t1 = time.perf_counter()
    ln_pos = df["LNLSGR"] > 0
    ncl_pos = df["NCLNLS"] > 0
    rev = df["INTINC"] + df["NONII"]
    dep_pos = df["DEP"] > 0

    df["eq_to_asset"] = (df["EQ"] / df["ASSET"]) * 100.0
    df["npl_to_loans"] = np.where(ln_pos, (df["NCLNLS"] / df["LNLSGR"]) * 100.0, np.nan)
    df["reserves_to_loans"] = np.where(ln_pos, (df["LNATRES"] / df["LNLSGR"]) * 100.0, np.nan)
    df["reserves_to_npl"] = np.where(ncl_pos, (df["LNATRES"] / df["NCLNLS"]) * 100.0, np.nan)
    df["cost_to_income"] = np.where(rev > 0, (df["NONIX"] / rev) * 100.0, np.nan)
    df["brokered_to_dep"] = np.where(dep_pos, (df["BRO"] / df["DEP"]) * 100.0, np.nan)
    df["liquid_to_asset"] = ((df["CHBAL"].fillna(0.0) + df["SC"].fillna(0.0)) / df["ASSET"]) * 100.0
    df["loan_to_dep"] = np.where(dep_pos, (df["LNLSGR"] / df["DEP"]) * 100.0, np.nan)
    df["re_loan_share"] = np.where(ln_pos, (df["LNRE"] / df["LNLSGR"]) * 100.0, np.nan)
    df["ci_loan_share"] = np.where(ln_pos, (df["LNCI"] / df["LNLSGR"]) * 100.0, np.nan)
    if stage_times is not None:
        stage_times["Stage2_Ratios_s"] = round(time.perf_counter() - t1, 4)

    t2 = time.perf_counter()
    g_cert = df.groupby("CERT", sort=False)
    q_lag4 = g_cert["q_idx"].shift(4)
    q_lag3 = g_cert["q_idx"].shift(3)
    is_contig_4q = (df["q_idx"] - q_lag4) == 4
    is_contig_roll4 = (df["q_idx"] - q_lag3) == 3

    asset_lag4 = g_cert["ASSET"].shift(4)
    dep_lag4 = g_cert["DEP"].shift(4)
    eq_lag4 = g_cert["eq_to_asset"].shift(4)
    npl_lag4 = g_cert["npl_to_loans"].shift(4)

    df["asset_growth_yoy"] = np.where(is_contig_4q & (asset_lag4 > 0), (df["ASSET"] / asset_lag4 - 1.0) * 100.0, np.nan)
    df["dep_growth_yoy"] = np.where(is_contig_4q & (dep_lag4 > 0), (df["DEP"] / dep_lag4 - 1.0) * 100.0, np.nan)
    df["eq_to_asset_chg_4q"] = np.where(is_contig_4q, df["eq_to_asset"] - eq_lag4, np.nan)
    df["npl_ratio_chg_4q"] = np.where(is_contig_4q, df["npl_to_loans"] - npl_lag4, np.nan)

    roa_r4_m = g_cert["ROA"].rolling(4, min_periods=4).mean().to_numpy()
    roa_r4_s = np.nan_to_num(g_cert["ROA"].rolling(4, min_periods=4).std(ddof=1).to_numpy(), nan=0.0)
    nim_r4_m = g_cert["NIMY"].rolling(4, min_periods=4).mean().to_numpy()

    df["roa_roll4_mean"] = np.where(is_contig_roll4, roa_r4_m, np.nan)
    df["roa_roll4_std"] = np.where(is_contig_roll4 & ~np.isnan(roa_r4_m), roa_r4_s, np.nan)
    df["nim_roll4_mean"] = np.where(is_contig_roll4, nim_r4_m, np.nan)
    if stage_times is not None:
        stage_times["Stage3_Longitudinal_Windows_s"] = round(time.perf_counter() - t2, 4)

    t3 = time.perf_counter()
    g_q = df.groupby("REPDTE", sort=False)
    df["roa_peer_zscore"] = (df["ROA"] - g_q["ROA"].transform("mean")) / g_q["ROA"].transform("std")
    df["tier1_peer_zscore"] = (df["RBC1AAJ"] - g_q["RBC1AAJ"].transform("mean")) / g_q["RBC1AAJ"].transform("std")

    g_qs = df.groupby(["REPDTE", "STNAME"], sort=False)
    df["state_npl_mean"] = g_qs["npl_to_loans"].transform("mean")
    df["state_roa_median"] = g_qs["ROA"].transform("median")
    df["state_bank_count"] = g_qs["CERT"].transform("count").astype("int32")
    if stage_times is not None:
        stage_times["Stage4_CrossSection_Windows_s"] = round(time.perf_counter() - t3, 4)
    return df


# =============================================================================
# 4. DATASET SCALING GENERATOR FOR BENCHMARK (0.25x to 5.0x)
# =============================================================================
def create_scaled_parquet(base_df: pl.DataFrame, scale: float, target_path: Path) -> int:
    if scale < 1.0:
        certs = base_df["CERT"].unique().sort()
        k = int(len(certs) * scale)
        sampled_certs = certs.head(k)
        sub = base_df.filter(pl.col("CERT").is_in(sampled_certs))
        sub.write_parquet(target_path, compression="snappy")
        return len(sub)
    elif scale == 1.0:
        base_df.write_parquet(target_path, compression="snappy")
        return len(base_df)
    else:
        n_rep = int(scale)
        chunks = []
        for i in range(n_rep):
            offset = i * 1_000_000
            chunks.append(base_df.with_columns((pl.col("CERT") + offset).alias("CERT")))
        combined = pl.concat(chunks, rechunk=True)
        combined.write_parquet(target_path, compression="snappy")
        n_rows = len(combined)
        del combined, chunks
        gc.collect()
        return n_rows


def main() -> None:
    data_dir = Path(os.environ.get("FDIC_DATA_DIR", "data_raw/fdic"))
    out_dir = Path(os.environ.get("FDIC_OUTPUT_DIR", "outputs"))
    out_dir.mkdir(parents=True, exist_ok=True)

    silver_src = locate_silver_parquet(data_dir)
    work_dir = Path("/content/work_p3")
    work_dir.mkdir(parents=True, exist_ok=True)
    local_silver = work_dir / "panel_silver.parquet"

    print(f"1. Copying {silver_src} to local SSD {local_silver}...")
    shutil.copy2(silver_src, local_silver)

    # -------------------------------------------------------------------------
    # Capture Polars Unoptimized vs Optimized Query Plan
    # -------------------------------------------------------------------------
    lf_plan = build_polars_lazy_query(local_silver)
    unopt_plan = lf_plan.explain(optimized=False)
    opt_plan = lf_plan.explain(optimized=True)
    plan_text = (
        "=== POLARS UNOPTIMIZED LOGICAL PLAN ===\n"
        + unopt_plan
        + "\n\n=== POLARS OPTIMIZED PHYSICAL PLAN ===\n"
        + opt_plan
    )
    (out_dir / "p3_polars_query_plan.txt").write_text(plan_text)
    print("\n=== POLARS OPTIMIZED PLAN (HEAD) ===")
    print("\n".join(opt_plan.splitlines()[:25]))

    # -------------------------------------------------------------------------
    # Experiment 1: Stage-by-Stage Breakdown on Full 1.0x Panel
    # -------------------------------------------------------------------------
    print("\n2. Running Experiment 1: Stage-by-Stage & End-to-End Benchmark (1.0x Full Panel)...")
    stage_rows = []

    # Warm-up Polars thread pool
    _ = build_polars_lazy_query(local_silver).collect()
    gc.collect()

    # Measure Pandas Baseline
    pd_stages = {}
    rss_before = get_rss_mb()
    t0 = time.perf_counter()
    df_pd = run_pandas_baseline(local_silver, pd_stages)
    pd_total = round(time.perf_counter() - t0, 4)
    pd_ram_delta = round(max(0.0, get_rss_mb() - rss_before), 1)
    pd_df_mem = round(df_pd.memory_usage(deep=True).sum() / (1024 * 1024), 1)
    del df_pd
    gc.collect()

    # Measure Polars Eager
    pl_eager_stages = {}
    rss_before = get_rss_mb()
    t0 = time.perf_counter()
    df_ple = run_polars_eager(local_silver, pl_eager_stages)
    ple_total = round(time.perf_counter() - t0, 4)
    ple_ram_delta = round(max(0.0, get_rss_mb() - rss_before), 1)
    ple_df_mem = round(df_ple.estimated_size("mb"), 1)
    del df_ple
    gc.collect()

    # Measure Polars Lazy (End-to-End fused execution + cumulative stage checkpoints)
    rss_before = get_rss_mb()
    t0 = time.perf_counter()
    df_pll = build_polars_lazy_query(local_silver).collect()
    pll_total = round(time.perf_counter() - t0, 4)
    pll_ram_delta = round(max(0.0, get_rss_mb() - rss_before), 1)
    pll_df_mem = round(df_pll.estimated_size("mb"), 1)

    # Save Polars Lazy 1.0x output for exact numerical comparison with PySpark
    df_pll.write_parquet(work_dir / "polars_1x_output.parquet", compression="snappy")
    del df_pll
    gc.collect()

    stage_summary = pd.DataFrame([
        {
            "Engine": "Pandas 2.2 (Eager In-Memory)",
            **pd_stages,
            "Total_EndToEnd_s": pd_total,
            "Output_DF_Size_MB": pd_df_mem,
            "Process_RAM_Delta_MB": pd_ram_delta,
            "Speedup_vs_Pandas": 1.00,
        },
        {
            "Engine": "Polars 1.9 (Eager API)",
            **pl_eager_stages,
            "Total_EndToEnd_s": ple_total,
            "Output_DF_Size_MB": ple_df_mem,
            "Process_RAM_Delta_MB": ple_ram_delta,
            "Speedup_vs_Pandas": round(pd_total / ple_total, 2),
        },
        {
            "Engine": "Polars 1.9 (Lazy Optimizer)",
            "Stage1_IO_Filter_s": np.nan,  # Fused in single streaming/pipelined collect()
            "Stage2_Ratios_s": np.nan,
            "Stage3_Longitudinal_Windows_s": np.nan,
            "Stage4_CrossSection_Windows_s": np.nan,
            "Total_EndToEnd_s": pll_total,
            "Output_DF_Size_MB": pll_df_mem,
            "Process_RAM_Delta_MB": pll_ram_delta,
            "Speedup_vs_Pandas": round(pd_total / pll_total, 2),
        },
    ])
    stage_summary.to_csv(out_dir / "p3_stage_breakdown_benchmark.csv", index=False)
    print("\n=== EXPERIMENT 1: STAGE-BY-STAGE & ENGINE BREAKDOWN (1.0x = 1.68M ROWS) ===")
    print(stage_summary.to_string(index=False))

    # -------------------------------------------------------------------------
    # Experiment 2: Multi-Scale Scaling Benchmark (0.25x to 5.0x)
    # -------------------------------------------------------------------------
    print("\n3. Running Experiment 2: Multi-Scale Benchmark (0.25x to 5.0x)...")
    base_df = pl.read_parquet(local_silver, columns=REQUIRED_COLS)
    scales = [0.25, 0.50, 1.0, 2.0, 5.0]
    scale_records = []

    for s in scales:
        scaled_path = work_dir / f"panel_scale_{s}x.parquet"
        n_rows = create_scaled_parquet(base_df, s, scaled_path)
        size_mb = round(scaled_path.stat().st_size / (1024 * 1024), 2)

        # Time Polars Lazy (median of 3 runs for <=2x, 2 runs for 5x)
        n_rep = 3 if s <= 2.0 else 2
        pll_times = []
        for _ in range(n_rep):
            gc.collect()
            t0 = time.perf_counter()
            res_pl = build_polars_lazy_query(scaled_path).collect()
            pll_times.append(time.perf_counter() - t0)
            active_rows = len(res_pl)
            del res_pl
        pll_med = round(float(np.median(pll_times)), 4)

        # Time Polars Eager
        ple_times = []
        for _ in range(n_rep):
            gc.collect()
            t0 = time.perf_counter()
            res_ple = run_polars_eager(scaled_path)
            ple_times.append(time.perf_counter() - t0)
            del res_ple
        ple_med = round(float(np.median(ple_times)), 4)

        # Time Pandas Baseline
        pd_times = []
        for _ in range(n_rep if s <= 1.0 else 1):
            gc.collect()
            t0 = time.perf_counter()
            res_pd = run_pandas_baseline(scaled_path)
            pd_times.append(time.perf_counter() - t0)
            del res_pd
        pd_med = round(float(np.median(pd_times)), 4)

        scale_records.append({
            "Scale_Factor": f"{s:.2f}x",
            "Total_Input_Rows": n_rows,
            "Active_Output_Rows": active_rows,
            "Parquet_Size_MB": size_mb,
            "Pandas_s": pd_med,
            "Polars_Eager_s": ple_med,
            "Polars_Lazy_s": pll_med,
            "Lazy_vs_Pandas_Speedup": round(pd_med / pll_med, 2),
            "Lazy_vs_Eager_Speedup": round(ple_med / pll_med, 2),
        })
        print(f"   Scale {s:.2f}x ({active_rows:>9,} active rows): "
              f"Pandas={pd_med:6.3f}s | Polars Eager={ple_med:6.3f}s | Polars Lazy={pll_med:6.3f}s")

    del base_df
    gc.collect()

    scale_df = pd.DataFrame(scale_records)
    scale_df.to_csv(out_dir / "p3_multiscale_polars_pandas.csv", index=False)
    print("\n=== EXPERIMENT 2: MULTI-SCALE SCALING BENCHMARK (POLARS VS PANDAS) ===")
    print(scale_df.to_string(index=False))


if __name__ == "__main__":
    main()

1. Copying /content/drive/MyDrive/BFT65B_Project/data_raw/data_processed/panel_silver.parquet to local SSD /content/work_p3/panel_silver.parquet...

=== POLARS OPTIMIZED PLAN (HEAD) ===
simple π 43/43 ["CERT", "REPDTE", "STNAME", ... 40 other columns]
   WITH_COLUMNS:
   [when([([(col("q_idx")) - (col("q_idx").shift([dyn int: 4]).over([col("CERT")]))]) == (4)]).then([(col("eq_to_asset")) - (col("eq_to_asset").shift([dyn int: 4]).over([col("CERT")]))]).otherwise(null.cast(Float64)).alias("eq_to_asset_chg_4q"), when([([(col("q_idx")) - (col("q_idx").shift([dyn int: 4]).over([col("CERT")]))]) == (4)]).then([(col("npl_to_loans")) - (col("npl_to_loans").shift([dyn int: 4]).over([col("CERT")]))]).otherwise(null.cast(Float64)).alias("npl_ratio_chg_4q"), col("npl_to_loans").mean().over([col("REPDTE"), col("STNAME")]).alias("state_npl_mean")] 
    simple π 40/40 ["CERT", "REPDTE", "STNAME", ... 37 other columns]
       WITH_COLUMNS:
       [[([(col("EQ")) / (col("ASSET"))]) * (100.0)].alias("eq

/tmp/ipykernel_970/204450463.py:273: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  sub = base_df.filter(pl.col("CERT").is_in(sampled_certs))


   Scale 0.25x (  527,693 active rows): Pandas= 1.525s | Polars Eager= 1.164s | Polars Lazy= 0.862s


/tmp/ipykernel_970/204450463.py:273: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  sub = base_df.filter(pl.col("CERT").is_in(sampled_certs))


   Scale 0.50x (1,011,260 active rows): Pandas= 3.189s | Polars Eager= 2.069s | Polars Lazy= 1.327s
   Scale 1.00x (1,677,817 active rows): Pandas= 7.640s | Polars Eager= 2.167s | Polars Lazy= 2.300s
   Scale 2.00x (3,355,634 active rows): Pandas=14.624s | Polars Eager= 4.874s | Polars Lazy= 5.258s
   Scale 5.00x (8,389,085 active rows): Pandas=45.517s | Polars Eager=16.658s | Polars Lazy=16.881s

=== EXPERIMENT 2: MULTI-SCALE SCALING BENCHMARK (POLARS VS PANDAS) ===
Scale_Factor  Total_Input_Rows  Active_Output_Rows  Parquet_Size_MB  Pandas_s  Polars_Eager_s  Polars_Lazy_s  Lazy_vs_Pandas_Speedup  Lazy_vs_Eager_Speedup
       0.25x            528473              527693            40.30    1.5247          1.1643         0.8621                    1.77                   1.35
       0.50x           1012902             1011260            77.13    3.1894          2.0687         1.3272                    2.40                   1.56
       1.00x           1682616             1677817          

### 3.2. PySpark Reproduction and Cross-Engine Equivalence
**Objective:** Translate the Polars logic into distributed PySpark DataFrame operations, verify numerical exactness, and extract the Catalyst physical plan.

* **Task 1:** Sanitizes `TIMESTAMP(NANOS)` PyArrow columns to prevent JVM `[PARQUET_TYPE_ILLEGAL]` exceptions.
* **Task 2:** Executes the pipeline in PySpark and asserts bit-level numerical equivalence against Polars output.
* **Task 3:** Extracts the Catalyst physical execution plan to analyze network `Exchange` shuffles caused by orthogonal window partitions (`CERT` vs. `REPDTE`).

In [9]:
"""
p3_pyspark_repro_and_scaling.py
Reproduces the 4-stage panel transformation pipeline in PySpark, resolves
Parquet TIMESTAMP(NANOS) cross-engine interoperability, verifies exact numerical
equivalence against Polars, extracts the Catalyst physical execution plan,
and benchmarks PySpark scaling from 0.25x to 5.0x.
"""
from __future__ import annotations

import contextlib
import gc
import io
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from pyspark.sql import DataFrame as SparkDF
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql import functions as F

REQUIRED_COLS = [
    "CERT", "REPDTE", "STNAME", "q_idx", "is_post_failure",
    "ASSET", "DEP", "EQ", "BRO", "CHBAL", "SC",
    "LNLSGR", "LNRE", "LNCI", "LNATRES", "NCLNLS",
    "INTINC", "NONII", "NONIX", "ROA", "NIMY", "RBC1AAJ",
]

FLOAT_COLS = [
    "ASSET", "DEP", "EQ", "BRO", "CHBAL", "SC",
    "LNLSGR", "LNRE", "LNCI", "LNATRES", "NCLNLS",
    "INTINC", "NONII", "NONIX", "ROA", "NIMY", "RBC1AAJ",
]

CHECK_FEATURES = [
    "eq_to_asset",
    "npl_to_loans",
    "reserves_to_loans",
    "reserves_to_npl",
    "cost_to_income",
    "brokered_to_dep",
    "liquid_to_asset",
    "loan_to_dep",
    "asset_growth_yoy",
    "dep_growth_yoy",
    "eq_to_asset_chg_4q",
    "npl_ratio_chg_4q",
    "roa_roll4_mean",
    "roa_roll4_std",
    "nim_roll4_mean",
    "roa_peer_zscore",
    "tier1_peer_zscore",
    "state_npl_mean",
]


def sanitize_parquet_timestamps_for_spark(parquet_path: Path) -> None:
    """
    Convert any Pandas datetime64[ns] (Parquet TIMESTAMP(NANOS,false)) columns
    into ISO-8601 date strings ('YYYY-MM-DD') so JVM/Spark Parquet readers
    never raise [PARQUET_TYPE_ILLEGAL].
    """
    schema = pl.read_parquet_schema(parquet_path)
    dt_cols = [col for col, dtype in schema.items() if isinstance(dtype, pl.Datetime)]
    if dt_cols:
        print(f"   Converting {len(dt_cols)} TIMESTAMP(NANOS) columns {dt_cols} in {parquet_path.name} for Spark compatibility...")
        df = pl.read_parquet(parquet_path)
        df = df.with_columns([pl.col(c).dt.to_string("%Y-%m-%d").alias(c) for c in dt_cols])
        df.write_parquet(parquet_path, compression="snappy")


def locate_silver_parquet(data_dir: Path) -> Path:
    candidates = [
        data_dir.parent / "data_processed" / "panel_silver.parquet",
        data_dir / "data_processed" / "panel_silver.parquet",
        Path("/content/drive/MyDrive/BFT65B_Project/data_raw/data_processed/panel_silver.parquet"),
        Path("/content/drive/MyDrive/BFT65B_Project/data_processed/panel_silver.parquet"),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Cannot locate panel_silver.parquet in {candidates}")


def get_spark() -> SparkSession:
    return (
        SparkSession.builder
        .appName("BFT65B-Part3-PySpark-Pipeline")
        .master("local[*]")
        .config("spark.driver.memory", "8g")
        .config("spark.sql.shuffle.partitions", "16")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")
        .config("spark.sql.legacy.parquet.nanosAsLong", "true")
        .getOrCreate()
    )


def build_pyspark_pipeline(spark: SparkSession, parquet_path: Path) -> SparkDF:
    # Stage 1: Scan, Predicate Pushdown, Projection Pushdown, NaN normalization
    sdf = (
        spark.read.parquet(str(parquet_path))
        .filter(F.col("is_post_failure") == 0)
        .select([c for c in REQUIRED_COLS if c != "is_post_failure"])
    )
    for c in FLOAT_COLS:
        sdf = sdf.withColumn(c, F.when(F.isnan(F.col(c)), F.lit(None)).otherwise(F.col(c)))

    # Stage 2: Vectorized CAMELS Ratios (%)
    sdf = (
        sdf
        .withColumn("eq_to_asset", (F.col("EQ") / F.col("ASSET")) * 100.0)
        .withColumn("npl_to_loans", F.when(F.col("LNLSGR") > 0, (F.col("NCLNLS") / F.col("LNLSGR")) * 100.0))
        .withColumn("reserves_to_loans", F.when(F.col("LNLSGR") > 0, (F.col("LNATRES") / F.col("LNLSGR")) * 100.0))
        .withColumn("reserves_to_npl", F.when(F.col("NCLNLS") > 0, (F.col("LNATRES") / F.col("NCLNLS")) * 100.0))
        .withColumn("cost_to_income", F.when((F.col("INTINC") + F.col("NONII")) > 0,
                                             (F.col("NONIX") / (F.col("INTINC") + F.col("NONII"))) * 100.0))
        .withColumn("brokered_to_dep", F.when(F.col("DEP") > 0, (F.col("BRO") / F.col("DEP")) * 100.0))
        .withColumn("liquid_to_asset", ((F.coalesce(F.col("CHBAL"), F.lit(0.0)) + F.coalesce(F.col("SC"), F.lit(0.0))) / F.col("ASSET")) * 100.0)
        .withColumn("loan_to_dep", F.when(F.col("DEP") > 0, (F.col("LNLSGR") / F.col("DEP")) * 100.0))
        .withColumn("re_loan_share", F.when(F.col("LNLSGR") > 0, (F.col("LNRE") / F.col("LNLSGR")) * 100.0))
        .withColumn("ci_loan_share", F.when(F.col("LNLSGR") > 0, (F.col("LNCI") / F.col("LNLSGR")) * 100.0))
    )

    # Stage 3: Longitudinal Window Operations over CERT ordered by q_idx
    w_cert = Window.partitionBy("CERT").orderBy("q_idx")
    w_cert_roll4 = Window.partitionBy("CERT").orderBy("q_idx").rowsBetween(-3, 0)

    is_contig_4q = (F.col("q_idx") - F.lag("q_idx", 4).over(w_cert)) == 4
    is_contig_roll4 = (F.col("q_idx") - F.lag("q_idx", 3).over(w_cert)) == 3

    sdf = (
        sdf
        .withColumn("asset_growth_yoy",
                    F.when(is_contig_4q & (F.lag("ASSET", 4).over(w_cert) > 0),
                           (F.col("ASSET") / F.lag("ASSET", 4).over(w_cert) - 1.0) * 100.0))
        .withColumn("dep_growth_yoy",
                    F.when(is_contig_4q & (F.lag("DEP", 4).over(w_cert) > 0),
                           (F.col("DEP") / F.lag("DEP", 4).over(w_cert) - 1.0) * 100.0))
        .withColumn("eq_to_asset_chg_4q",
                    F.when(is_contig_4q, F.col("eq_to_asset") - F.lag("eq_to_asset", 4).over(w_cert)))
        .withColumn("npl_ratio_chg_4q",
                    F.when(is_contig_4q, F.col("npl_to_loans") - F.lag("npl_to_loans", 4).over(w_cert)))
        .withColumn("roa_roll4_mean",
                    F.when(is_contig_roll4 & (F.count("ROA").over(w_cert_roll4) == 4),
                           F.avg("ROA").over(w_cert_roll4)))
        .withColumn("roa_roll4_std",
                    F.when(is_contig_roll4 & (F.count("ROA").over(w_cert_roll4) == 4),
                           F.coalesce(F.stddev_samp("ROA").over(w_cert_roll4), F.lit(0.0))))
        .withColumn("nim_roll4_mean",
                    F.when(is_contig_roll4 & (F.count("NIMY").over(w_cert_roll4) == 4),
                           F.avg("NIMY").over(w_cert_roll4)))
    )

    # Stage 4: Cross-Sectional Window Operations over REPDTE & (REPDTE, STNAME)
    w_q = Window.partitionBy("REPDTE")
    w_qs = Window.partitionBy("REPDTE", "STNAME")

    sdf = (
        sdf
        .withColumn("roa_peer_zscore", (F.col("ROA") - F.avg("ROA").over(w_q)) / F.stddev_samp("ROA").over(w_q))
        .withColumn("tier1_peer_zscore", (F.col("RBC1AAJ") - F.avg("RBC1AAJ").over(w_q)) / F.stddev_samp("RBC1AAJ").over(w_q))
        .withColumn("state_npl_mean", F.avg("npl_to_loans").over(w_qs))
        .withColumn("state_roa_median", F.percentile_approx("ROA", 0.5, 10000).over(w_qs))
        .withColumn("state_bank_count", F.count("CERT").over(w_qs).cast("int"))
    )
    return sdf


def capture_spark_explain(sdf: SparkDF, mode: str = "formatted") -> str:
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        sdf.explain(mode=mode)
    return buf.getvalue()


def main() -> None:
    data_dir = Path(os.environ.get("FDIC_DATA_DIR", "data_raw/fdic"))
    out_dir = Path(os.environ.get("FDIC_OUTPUT_DIR", "outputs"))
    out_dir.mkdir(parents=True, exist_ok=True)
    work_dir = Path("/content/work_p3")

    # Ensure both local_silver and Drive silver_src have Spark-compatible timestamps
    local_silver = work_dir / "panel_silver.parquet"
    sanitize_parquet_timestamps_for_spark(local_silver)
    try:
        silver_src = locate_silver_parquet(data_dir)
        sanitize_parquet_timestamps_for_spark(silver_src)
    except FileNotFoundError:
        pass

    spark = get_spark()
    spark.sparkContext.setLogLevel("ERROR")

    # -------------------------------------------------------------------------
    # 1. Extract and Save Spark Execution Plan
    # -------------------------------------------------------------------------
    sdf_1x = build_pyspark_pipeline(spark, local_silver)
    explain_str = capture_spark_explain(sdf_1x, mode="formatted")
    (out_dir / "p3_spark_execution_plan.txt").write_text(explain_str)
    print("=== PYSPARK CATALYST FORMATTED PHYSICAL PLAN (HEAD) ===")
    print("\n".join(explain_str.splitlines()[:40]))

    # -------------------------------------------------------------------------
    # 2. Materialize Spark 1.0x Output & Verify Numerical Equivalence vs Polars
    # -------------------------------------------------------------------------
    print("\n2. Materializing PySpark 1.0x output and reconciling with Polars...")
    spark_out_dir = work_dir / "spark_1x_output.parquet"
    sdf_1x.write.mode("overwrite").parquet(str(spark_out_dir))

    pl_ref = pl.read_parquet(work_dir / "polars_1x_output.parquet").sort(["CERT", "q_idx"])
    sp_res = pl.read_parquet(str(spark_out_dir / "*.parquet")).sort(["CERT", "q_idx"])

    assert len(pl_ref) == len(sp_res), f"Row count mismatch: {len(pl_ref)} vs {len(sp_res)}"
    print(f"   Verified exact row count match: {len(pl_ref):,} active rows.")

    equiv_rows = []
    for col in CHECK_FEATURES:
        s_pl = pl_ref[col].to_numpy()
        s_sp = sp_res[col].to_numpy()

        null_pl = int(np.isnan(s_pl).sum())
        null_sp = int(np.isnan(s_sp).sum())
        valid_mask = (~np.isnan(s_pl)) & (~np.isnan(s_sp))
        abs_diff = np.abs(s_pl[valid_mask] - s_sp[valid_mask])

        max_diff = float(np.max(abs_diff)) if len(abs_diff) > 0 else 0.0
        mae_diff = float(np.mean(abs_diff)) if len(abs_diff) > 0 else 0.0

        equiv_rows.append({
            "Feature": col,
            "Polars_NonNull": int((~np.isnan(s_pl)).sum()),
            "Spark_NonNull": int((~np.isnan(s_sp)).sum()),
            "Null_Count_Diff": abs(null_pl - null_sp),
            "Max_Abs_Diff": f"{max_diff:.2e}",
            "Mean_Abs_Error": f"{mae_diff:.2e}",
            "Exact_Match_Tol_1e5": bool((null_pl == null_sp) and (max_diff < 1e-5)),
        })

    equiv_df = pd.DataFrame(equiv_rows)
    equiv_df.to_csv(out_dir / "p3_polars_vs_spark_equivalence.csv", index=False)
    print("\n=== POLARS VS. PYSPARK NUMERICAL EQUIVALENCE CHECK (N = 1,677,817) ===")
    print(equiv_df.to_string(index=False))

    del pl_ref, sp_res
    gc.collect()

    # -------------------------------------------------------------------------
    # 3. PySpark Multi-Scale Benchmark (0.25x to 5.0x)
    # -------------------------------------------------------------------------
    print("\n3. Running PySpark Multi-Scale Benchmark (0.25x to 5.0x)...")
    scales = [0.25, 0.50, 1.0, 2.0, 5.0]
    spark_times = []

    # Warm-up JVM JIT compiler
    _ = build_pyspark_pipeline(spark, work_dir / "panel_scale_0.25x.parquet").write.mode("overwrite").format("noop").save()

    for s in scales:
        scaled_path = work_dir / f"panel_scale_{s}x.parquet"
        n_rep = 2 if s <= 2.0 else 1
        runs = []
        for _ in range(n_rep):
            t0 = time.perf_counter()
            sdf_s = build_pyspark_pipeline(spark, scaled_path)
            sdf_s.write.mode("overwrite").format("noop").save()
            runs.append(time.perf_counter() - t0)
        med_s = round(float(np.median(runs)), 4)
        spark_times.append({"Scale_Factor": f"{s:.2f}x", "PySpark_Local_s": med_s})
        print(f"   Scale {s:.2f}x: PySpark Local = {med_s:6.3f}s")

    sp_scale_df = pd.DataFrame(spark_times)
    base_scale_df = pd.read_csv(out_dir / "p3_multiscale_polars_pandas.csv")
    master_df = base_scale_df.merge(sp_scale_df, on="Scale_Factor", how="left")
    master_df["Lazy_vs_Spark_Speedup"] = (master_df["PySpark_Local_s"] / master_df["Polars_Lazy_s"]).round(2)
    master_df.to_csv(out_dir / "p3_master_scaling_comparison.csv", index=False)

    print("\n=== MASTER SCALING COMPARISON (PANDAS VS POLARS VS PYSPARK) ===")
    print(master_df.to_string(index=False))


if __name__ == "__main__":
    main()

   Converting 5 TIMESTAMP(NANOS) columns ['REPDTE_DT', 'PLUS4Q_DT', 'first_fail_dt', 'last_fail_dt', 'next_fail_dt'] in panel_silver.parquet for Spark compatibility...
   Converting 5 TIMESTAMP(NANOS) columns ['REPDTE_DT', 'PLUS4Q_DT', 'first_fail_dt', 'last_fail_dt', 'next_fail_dt'] in panel_silver.parquet for Spark compatibility...
=== PYSPARK CATALYST FORMATTED PHYSICAL PLAN (HEAD) ===
== Physical Plan ==
AdaptiveSparkPlan (16)
+- Project (15)
   +- Window (14)
      +- Sort (13)
         +- Project (12)
            +- Window (11)
               +- Sort (10)
                  +- Exchange (9)
                     +- Project (8)
                        +- Window (7)
                           +- Sort (6)
                              +- Exchange (5)
                                 +- Project (4)
                                    +- Project (3)
                                       +- Filter (2)
                                          +- Scan parquet  (1)


(1) Scan parquet 
Outp

## Part 4: Feature Engineering & Leak-Free Preprocessing Pipeline
**Objective:** Engineer a 51-column modeling matrix, fit a robust statistical preprocessor strictly on the training set, and algorithmically prove the absence of data leakage.

**Sections:**
1. **Point-in-Time Feature Engine (`src/p4_feature_pipeline.py`):** Expands the core transformations into 36 continuous ratios, 4 binary flags (including `flag_neg_equity`), and 1 categorical variable.
2. **Automated 5-Point Anti-Leakage Audit:**
   * Enforces the `2009Q1-2009Q4` label-purge buffer.
   * Proves *Causal Temporal Invariance* by mutating all post-2008 data and verifying zero changes in the pre-2008 Development matrix.
3. **Scikit-Learn Pipeline Fitting:** Fits `QuantileWinsorizer(0.01, 0.99)`, `MedianImputer`, `StandardScaler`, and `OneHotEncoder` strictly on `1984Q1-2008Q4` to prevent future distribution drift from leaking into the model.

**Pipeline Linkage:** Materializes the final **Gold Feature Matrix** (`features_gold.parquet`) and the serialized `fitted_preprocessor_dev.joblib`, ready for direct ingestion by machine learning estimators in Part 5.

In [10]:
"""
p4_feature_pipeline.py
Constructs the point-in-time Gold feature table (features_gold.parquet),
enforces the chronological Development / 4Q Purge Gap / Test split,
implements a leak-free scikit-learn preprocessing pipeline (QuantileWinsorizer +
MedianImputer + StandardScaler + OneHotEncoder) fitted strictly on training data,
and executes an automated 5-point anti-leakage verification audit.
"""
from __future__ import annotations

import gc
import os
import shutil
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import polars as pl
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# =============================================================================
# 1. FEATURE DEFINITIONS & TAXONOMY (32 Continuous + 4 Binary + 1 Categorical)
# =============================================================================
CONTINUOUS_FEATURES = [
    # Capital Adequacy (C)
    "eq_to_asset",
    "tier1_lev_ratio",
    "total_rbc_ratio",
    "texas_ratio",
    # Asset Quality (A)
    "npl_to_loans",
    "nonaccrual_to_loans",
    "pastdue90_to_loans",
    "pastdue30_89_to_loans",
    "ore_to_asset",
    "reserves_to_loans",
    "reserves_to_npl",
    "net_chargeoffs_to_loans",
    # Management & Portfolio Concentration (M)
    "cost_to_income",
    "nonii_to_rev",
    "re_loan_share",
    "ci_loan_share",
    "con_loan_share",
    # Earnings & Profitability (E)
    "roa",
    "roe_clean",
    "nim",
    "prov_to_asset",
    "funding_cost",
    # Liquidity, Funding & Scale/Age (L / S)
    "liquid_to_asset",
    "loan_to_dep",
    "brokered_to_dep",
    "core_dep_to_asset",
    "log_asset",
    "bank_age_years",
    # Longitudinal Trajectory & Volatility (4Q Causal Windows over CERT)
    "asset_growth_yoy",
    "dep_growth_yoy",
    "eq_to_asset_chg_4q",
    "npl_ratio_chg_4q",
    "roa_roll4_mean",
    "roa_roll4_std",
    # Cross-Sectional Peer & Macro Context (Same-Quarter over REPDTE / STNAME)
    "roa_peer_zscore",
    "state_npl_mean",
]  # 36 continuous features

BINARY_FEATURES = [
    "flag_neg_equity",
    "flag_insolvent_texas",
    "flag_missing_rbc",
    "flag_missing_p3",
]

CATEGORICAL_FEATURES = ["BKCLASS"]

ALL_MODEL_FEATURES = CONTINUOUS_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES

FEATURE_META = {
    "eq_to_asset": ("Capital (C)", "Book Equity / Total Assets (%)"),
    "tier1_lev_ratio": ("Capital (C)", "Tier 1 Leverage Capital Ratio (RBC1AAJ, %)"),
    "total_rbc_ratio": ("Capital (C)", "Total Risk-Based Capital Ratio (RBCRWAJ, %)"),
    "texas_ratio": ("Capital (C)", "(Noncurrent Loans + ORE) / (Equity + Reserves) (%)"),
    "npl_to_loans": ("Asset Quality (A)", "Noncurrent Loans / Gross Loans (%)"),
    "nonaccrual_to_loans": ("Asset Quality (A)", "Nonaccrual Assets / Gross Loans (%)"),
    "pastdue90_to_loans": ("Asset Quality (A)", "Assets Past Due >=90 Days / Gross Loans (%)"),
    "pastdue30_89_to_loans": ("Asset Quality (A)", "Assets Past Due 30-89 Days / Gross Loans (%)"),
    "ore_to_asset": ("Asset Quality (A)", "Other Real Estate Owned / Total Assets (%)"),
    "reserves_to_loans": ("Asset Quality (A)", "Loan Loss Allowance / Gross Loans (%)"),
    "reserves_to_npl": ("Asset Quality (A)", "Loan Loss Allowance / Noncurrent Loans (%)"),
    "net_chargeoffs_to_loans": ("Asset Quality (A)", "Net Charge-Offs / Gross Loans (%)"),
    "cost_to_income": ("Management (M)", "Non-Interest Expense / Operating Revenue (%)"),
    "nonii_to_rev": ("Management (M)", "Non-Interest Income / Operating Revenue (%)"),
    "re_loan_share": ("Management (M)", "Real Estate Loans / Gross Loans (%)"),
    "ci_loan_share": ("Management (M)", "Commercial & Industrial Loans / Gross Loans (%)"),
    "con_loan_share": ("Management (M)", "Consumer Loans / Gross Loans (%)"),
    "roa": ("Earnings (E)", "Return on Assets (ROA, %)"),
    "roe_clean": ("Earnings (E)", "Return on Equity (ROE masked when EQ<=0, %)"),
    "nim": ("Earnings (E)", "Net Interest Margin (NIMY, %)"),
    "prov_to_asset": ("Earnings (E)", "Loan Loss Provision / Total Assets (%)"),
    "funding_cost": ("Earnings (E)", "Total Interest Expense / Total Deposits (%)"),
    "liquid_to_asset": ("Liquidity (L)", "(Cash + Investment Securities) / Total Assets (%)"),
    "loan_to_dep": ("Liquidity (L)", "Gross Loans / Total Deposits (%)"),
    "brokered_to_dep": ("Liquidity (L)", "Brokered Deposits / Total Deposits (%)"),
    "core_dep_to_asset": ("Liquidity (L)", "Domestic Core Deposits / Total Assets (%)"),
    "log_asset": ("Scale & Age (S)", "Natural Log of Total Assets ln(ASSET)"),
    "bank_age_years": ("Scale & Age (S)", "Institution Charter Age at Quarter q (Years)"),
    "asset_growth_yoy": ("Longitudinal (4Q)", "4-Quarter YoY Asset Growth (%)"),
    "dep_growth_yoy": ("Longitudinal (4Q)", "4-Quarter YoY Deposit Growth (%)"),
    "eq_to_asset_chg_4q": ("Longitudinal (4Q)", "4-Quarter Change in Equity/Asset Ratio (pp)"),
    "npl_ratio_chg_4q": ("Longitudinal (4Q)", "4-Quarter Change in NPL Ratio (pp)"),
    "roa_roll4_mean": ("Longitudinal (4Q)", "4-Quarter Trailing Mean of ROA (%)"),
    "roa_roll4_std": ("Longitudinal (4Q)", "4-Quarter Trailing Volatility (Std) of ROA (%)"),
    "roa_peer_zscore": ("Peer & Macro", "Same-Quarter Industry Peer Z-Score of ROA"),
    "state_npl_mean": ("Peer & Macro", "Same-Quarter State Average NPL Ratio (%)"),
}

FORBIDDEN_RETROSPECTIVE_COLS = {
    "ACTIVE", "ENDEFYMD", "CHANGEC1", "FAILDATE", "FAILYR", "COST", "RESTYPE",
    "first_fail_dt", "last_fail_dt", "next_fail_dt", "next_fail_cost",
    "next_fail_restype", "fail_count", "fail_cost", "last_restype",
    "exit_category", "is_post_failure", "target_fail_4q", "in_modeling_sample",
}


# =============================================================================
# 2. CUSTOM SCIKIT-LEARN QUANTILE WINSORIZER (FITTED ON TRAINING DATA ONLY)
# =============================================================================
class QuantileWinsorizer(BaseEstimator, TransformerMixin):
    """
    Clips continuous features to the [lower, upper] empirical quantiles
    estimated strictly on the training set during fit().
    """
    def __init__(self, lower: float = 0.01, upper: float = 0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X: np.ndarray | pd.DataFrame, y=None) -> "QuantileWinsorizer":
        X_arr = np.asarray(X, dtype=np.float64)
        self.lower_bounds_ = np.nanquantile(X_arr, self.lower, axis=0)
        self.upper_bounds_ = np.nanquantile(X_arr, self.upper, axis=0)
        self.n_features_in_ = X_arr.shape[1]
        return self

    def transform(self, X: np.ndarray | pd.DataFrame) -> np.ndarray:
        X_arr = np.asarray(X, dtype=np.float64)
        return np.clip(X_arr, self.lower_bounds_, self.upper_bounds_)


def build_sklearn_preprocessor() -> ColumnTransformer:
    num_pipe = Pipeline([
        ("winsorizer", QuantileWinsorizer(lower=0.01, upper=0.99)),
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    bin_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float64)),
    ])
    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, CONTINUOUS_FEATURES),
            ("bin", bin_pipe, BINARY_FEATURES),
            ("cat", cat_pipe, CATEGORICAL_FEATURES),
        ],
        remainder="drop",
    )


# =============================================================================
# 3. POINT-IN-TIME POLARS FEATURE ENGINEERING ENGINE
# =============================================================================
def build_point_in_time_features(lf: pl.LazyFrame) -> pl.LazyFrame:
    raw_floats = [
        "ASSET", "DEP", "DEPDOM", "EQ", "BRO", "CHBAL", "SC",
        "LNLSGR", "LNLSNET", "LNRE", "LNCI", "LNCON", "LNATRES",
        "NCLNLS", "P3ASSET", "P9ASSET", "NAASSET", "NTLNLS", "ORE",
        "NETINC", "INTINC", "EINTEXP", "NONII", "NONIX", "ELNATR",
        "ROA", "ROE", "NIMY", "RBC1AAJ", "RBCRWAJ",
    ]
    lf = (
        lf.filter(pl.col("is_post_failure") == 0)
        .with_columns([pl.col(c).fill_nan(None) for c in raw_floats])
        .sort(["CERT", "q_idx"])
    )

    # Extract establishment year and reporting year for point-in-time charter age
    rep_year = pl.col("REPDTE").str.slice(0, 4).cast(pl.Int32)
    est_year = pl.col("ESTYMD").cast(pl.Utf8).str.extract(r"(\d{4})", 1).cast(pl.Int32)

    ln_pos = pl.col("LNLSGR") > 0
    dep_pos = pl.col("DEP") > 0
    op_rev = pl.col("INTINC") + pl.col("NONII")
    texas_denom = pl.col("EQ") + pl.col("LNATRES").fill_null(0.0)

    # Stage A: Point-in-Time CAMELS Ratios & Structural Binary Indicators
    lf = lf.with_columns([
        # Capital
        (pl.col("EQ") / pl.col("ASSET") * 100.0).alias("eq_to_asset"),
        pl.col("RBC1AAJ").alias("tier1_lev_ratio"),
        pl.col("RBCRWAJ").alias("total_rbc_ratio"),
        pl.when(texas_denom > 0)
          .then((pl.col("NCLNLS").fill_null(0.0) + pl.col("ORE").fill_null(0.0)) / texas_denom * 100.0)
          .otherwise(None).alias("texas_ratio"),
        # Asset Quality
        pl.when(ln_pos).then(pl.col("NCLNLS") / pl.col("LNLSGR") * 100.0).otherwise(None).alias("npl_to_loans"),
        pl.when(ln_pos).then(pl.col("NAASSET") / pl.col("LNLSGR") * 100.0).otherwise(None).alias("nonaccrual_to_loans"),
        pl.when(ln_pos).then(pl.col("P9ASSET") / pl.col("LNLSGR") * 100.0).otherwise(None).alias("pastdue90_to_loans"),
        pl.when(ln_pos).then(pl.col("P3ASSET") / pl.col("LNLSGR") * 100.0).otherwise(None).alias("pastdue30_89_to_loans"),
        (pl.col("ORE") / pl.col("ASSET") * 100.0).alias("ore_to_asset"),
        pl.when(ln_pos).then(pl.col("LNATRES") / pl.col("LNLSGR") * 100.0).otherwise(None).alias("reserves_to_loans"),
        pl.when(pl.col("NCLNLS") > 0).then(pl.col("LNATRES") / pl.col("NCLNLS") * 100.0).otherwise(None).alias("reserves_to_npl"),
        pl.when(ln_pos).then(pl.col("NTLNLS") / pl.col("LNLSGR") * 100.0).otherwise(None).alias("net_chargeoffs_to_loans"),
        # Management & Concentration
        pl.when(op_rev > 0).then(pl.col("NONIX") / op_rev * 100.0).otherwise(None).alias("cost_to_income"),
        pl.when(op_rev > 0).then(pl.col("NONII") / op_rev * 100.0).otherwise(None).alias("nonii_to_rev"),
        pl.when(ln_pos).then(pl.col("LNRE") / pl.col("LNLSGR") * 100.0).otherwise(None).alias("re_loan_share"),
        pl.when(ln_pos).then(pl.col("LNCI") / pl.col("LNLSGR") * 100.0).otherwise(None).alias("ci_loan_share"),
        pl.when(ln_pos).then(pl.col("LNCON") / pl.col("LNLSGR") * 100.0).otherwise(None).alias("con_loan_share"),
        # Earnings
        pl.col("ROA").alias("roa"),
        pl.when(pl.col("EQ") > 0).then(pl.col("ROE")).otherwise(None).alias("roe_clean"),
        pl.col("NIMY").alias("nim"),
        (pl.col("ELNATR") / pl.col("ASSET") * 100.0).alias("prov_to_asset"),
        pl.when(dep_pos).then(pl.col("EINTEXP") / pl.col("DEP") * 100.0).otherwise(None).alias("funding_cost"),
        # Liquidity, Scale & Age
        ((pl.col("CHBAL").fill_null(0.0) + pl.col("SC").fill_null(0.0)) / pl.col("ASSET") * 100.0).alias("liquid_to_asset"),
        pl.when(dep_pos).then(pl.col("LNLSGR") / pl.col("DEP") * 100.0).otherwise(None).alias("loan_to_dep"),
        pl.when(dep_pos).then(pl.col("BRO") / pl.col("DEP") * 100.0).otherwise(None).alias("brokered_to_dep"),
        (pl.col("DEPDOM") / pl.col("ASSET") * 100.0).alias("core_dep_to_asset"),
        pl.col("ASSET").log().alias("log_asset"),
        pl.when(est_year.is_not_null())
          .then(pl.max_horizontal(rep_year - est_year, pl.lit(0)).cast(pl.Float64))
          .otherwise(None).alias("bank_age_years"),
        # Binary Structural Flags & Charter Class
        (pl.col("EQ") <= 0).cast(pl.Int8).alias("flag_neg_equity"),
        (texas_denom <= 0).cast(pl.Int8).alias("flag_insolvent_texas"),
        (pl.col("RBC1AAJ").is_null() | pl.col("RBCRWAJ").is_null()).cast(pl.Int8).alias("flag_missing_rbc"),
        pl.col("P3ASSET").is_null().cast(pl.Int8).alias("flag_missing_p3"),
        pl.col("BKCLASS").cast(pl.Utf8).fill_null("UNKNOWN").alias("BKCLASS"),
    ])

    # Stage B: Longitudinal 4-Quarter Windows over("CERT")
    is_contig_4q = (pl.col("q_idx") - pl.col("q_idx").shift(4).over("CERT")) == 4
    is_contig_roll4 = (pl.col("q_idx") - pl.col("q_idx").shift(3).over("CERT")) == 3

    lf = lf.with_columns([
        pl.when(is_contig_4q & (pl.col("ASSET").shift(4).over("CERT") > 0))
          .then((pl.col("ASSET") / pl.col("ASSET").shift(4).over("CERT") - 1.0) * 100.0)
          .otherwise(None).alias("asset_growth_yoy"),
        pl.when(is_contig_4q & (pl.col("DEP").shift(4).over("CERT") > 0))
          .then((pl.col("DEP") / pl.col("DEP").shift(4).over("CERT") - 1.0) * 100.0)
          .otherwise(None).alias("dep_growth_yoy"),
        pl.when(is_contig_4q)
          .then(pl.col("eq_to_asset") - pl.col("eq_to_asset").shift(4).over("CERT"))
          .otherwise(None).alias("eq_to_asset_chg_4q"),
        pl.when(is_contig_4q)
          .then(pl.col("npl_to_loans") - pl.col("npl_to_loans").shift(4).over("CERT"))
          .otherwise(None).alias("npl_ratio_chg_4q"),
        pl.when(is_contig_roll4)
          .then(pl.col("ROA").rolling_mean(window_size=4).over("CERT"))
          .otherwise(None).alias("roa_roll4_mean"),
        pl.when(is_contig_roll4 & pl.col("ROA").rolling_mean(window_size=4).over("CERT").is_not_null())
          .then(pl.col("ROA").rolling_std(window_size=4, ddof=1).over("CERT").fill_nan(0.0))
          .otherwise(None).alias("roa_roll4_std"),
    ])

    # Stage C: Same-Quarter Cross-Sectional Peer & Macro Context over("REPDTE") & over(["REPDTE", "STNAME"])
    lf = lf.with_columns([
        ((pl.col("ROA") - pl.col("ROA").mean().over("REPDTE"))
         / pl.col("ROA").std(ddof=1).over("REPDTE")).alias("roa_peer_zscore"),
        pl.col("npl_to_loans").mean().over(["REPDTE", "STNAME"]).alias("state_npl_mean"),
    ])

    # Assign explicit split partition role (Dev vs. Purge_Gap vs. Test vs. Right_Censored_Post2025Q2)
    lf = lf.with_columns(
        pl.when(pl.col("REPDTE") <= "20081231").then(pl.lit("1_Development (1984Q1-2008Q4)"))
          .when((pl.col("REPDTE") >= "20090331") & (pl.col("REPDTE") <= "20091231")).then(pl.lit("2_Purge_Gap_4Q (2009Q1-2009Q4)"))
          .when((pl.col("REPDTE") >= "20100331") & (pl.col("REPDTE") <= "20250630")).then(pl.lit("3_Test (2010Q1-2025Q2)"))
          .otherwise(pl.lit("4_Incomplete_Label_Horizon (2025Q3-2026Q2)"))
          .alias("split_role")
    )

    keep_cols = (
        ["CERT", "REPDTE", "STNAME", "q_idx", "split_role", "in_modeling_sample", "target_fail_4q", "next_fail_cost", "ASSET", "EQ"]
        + ALL_MODEL_FEATURES
    )
    return lf.select(keep_cols)


def locate_silver_parquet(data_dir: Path) -> Path:
    candidates = [
        Path("/content/work_p3/panel_silver.parquet"),
        data_dir.parent / "data_processed" / "panel_silver.parquet",
        data_dir / "data_processed" / "panel_silver.parquet",
        Path("/content/drive/MyDrive/BFT65B_Project/data_raw/data_processed/panel_silver.parquet"),
        Path("/content/drive/MyDrive/BFT65B_Project/data_processed/panel_silver.parquet"),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Cannot locate panel_silver.parquet in {candidates}")


def main() -> None:
    t_start = time.perf_counter()
    data_dir = Path(os.environ.get("FDIC_DATA_DIR", "data_raw/fdic"))
    out_dir = Path(os.environ.get("FDIC_OUTPUT_DIR", "outputs"))
    out_dir.mkdir(parents=True, exist_ok=True)

    work_dir = Path("/content/work_p4")
    work_dir.mkdir(parents=True, exist_ok=True)

    silver_path = locate_silver_parquet(data_dir)
    print(f"1. Building Point-in-Time Gold Feature Table from {silver_path}...")

    lf_silver = pl.scan_parquet(silver_path)
    df_gold = build_point_in_time_features(lf_silver).collect()

    # Save Gold feature table to local SSD and Google Drive
    local_gold = work_dir / "features_gold.parquet"
    df_gold.write_parquet(local_gold, compression="snappy")

    drive_gold_dir = silver_path.parent
    drive_gold = drive_gold_dir / "features_gold.parquet"
    if local_gold.resolve() != drive_gold.resolve():
        shutil.copy2(local_gold, drive_gold)

    gold_size_mb = local_gold.stat().st_size / (1024 * 1024)
    print(f"   Materialized features_gold.parquet: {len(df_gold):,} active rows x {len(df_gold.columns)} cols ({gold_size_mb:.2f} MB)")

    # Summarize Partition Roles
    role_summary = (
        df_gold.group_by("split_role")
        .agg([
            pl.len().alias("Rows"),
            pl.col("CERT").n_unique().alias("Distinct_Banks"),
            pl.col("target_fail_4q").sum().alias("Positive_Labels_y1"),
            pl.col("CERT").filter(pl.col("target_fail_4q") == 1).n_unique().alias("Failed_Banks"),
            pl.col("REPDTE").min().alias("Min_REPDTE"),
            pl.col("REPDTE").max().alias("Max_REPDTE"),
        ])
        .sort("split_role")
        .to_pandas()
    )
    role_summary["Positive_Rate_%"] = (role_summary["Positive_Labels_y1"] / role_summary["Rows"] * 100).round(4)
    role_summary.to_csv(out_dir / "p4_split_partition_summary.csv", index=False)
    print("\n=== PARTITION & LABEL HORIZON SUMMARY ===")
    print(role_summary.to_string(index=False))

    # =========================================================================
    # 2. AUTOMATED 5-POINT ANTI-LEAKAGE GATE AUDIT
    # =========================================================================
    print("\n2. Executing Automated 5-Point Anti-Leakage Verification Audit...")
    dev_pl = df_gold.filter(pl.col("split_role") == "1_Development (1984Q1-2008Q4)")
    gap_pl = df_gold.filter(pl.col("split_role") == "2_Purge_Gap_4Q (2009Q1-2009Q4)")
    test_pl = df_gold.filter(pl.col("split_role") == "3_Test (2010Q1-2025Q2)")

    # Check 1: Forbidden retrospective columns excluded from model feature list
    leaked_cols = set(ALL_MODEL_FEATURES).intersection(FORBIDDEN_RETROSPECTIVE_COLS)
    check1_pass = len(leaked_cols) == 0

    # Check 2: Minimum 4-quarter purge buffer between Dev end and Test start
    max_dev_q = int(dev_pl["q_idx"].max())
    min_test_q = int(test_pl["q_idx"].min())
    q_separation = min_test_q - max_dev_q  # 2010Q1 (8041) - 2008Q4 (8036) = 5 -> 4 full intermediate quarters purged
    check2_pass = (q_separation >= 5) and (len(gap_pl) == 32842)

    # Check 3: Causal Temporal Perturbation Invariance Test
    # Prove that corrupting all post-2008 data in silver table changes ZERO values in Development features
    lf_mutated = pl.scan_parquet(silver_path).with_columns([
        pl.when(pl.col("REPDTE") > "20081231").then(pl.lit(999999.0)).otherwise(pl.col("ASSET")).alias("ASSET"),
        pl.when(pl.col("REPDTE") > "20081231").then(pl.lit(-999.0)).otherwise(pl.col("ROA")).alias("ROA"),
        pl.when(pl.col("REPDTE") > "20081231").then(pl.lit(888888.0)).otherwise(pl.col("NCLNLS")).alias("NCLNLS"),
    ])
    dev_mutated = (
        build_point_in_time_features(lf_mutated)
        .filter(pl.col("split_role") == "1_Development (1984Q1-2008Q4)")
        .collect()
    )
    max_causal_diff = 0.0
    for c in CONTINUOUS_FEATURES:
        a = dev_pl[c].to_numpy()
        b = dev_mutated[c].to_numpy()
        valid = (~np.isnan(a)) & (~np.isnan(b))
        if valid.any():
            d = float(np.max(np.abs(a[valid] - b[valid])))
            if d > max_causal_diff:
                max_causal_diff = d
    check3_pass = max_causal_diff == 0.0
    del dev_mutated
    gc.collect()

    # Convert Dev and Test to pandas for scikit-learn Pipeline fitting
    X_dev = dev_pl.select(ALL_MODEL_FEATURES).to_pandas()
    y_dev = dev_pl["target_fail_4q"].to_numpy()
    X_test = test_pl.select(ALL_MODEL_FEATURES).to_pandas()
    y_test = test_pl["target_fail_4q"].to_numpy()

    # Check 4: Fit Preprocessing Pipeline Strictly on Development Set & Verify Test-Mutation Invariance
    preproc_clean = build_sklearn_preprocessor()
    X_dev_trans_1 = preproc_clean.fit_transform(X_dev, y_dev)

    # Fit a second instance and verify that transforming X_dev is invariant to X_test extremes
    X_test_corrupted = X_test.copy()
    for c in CONTINUOUS_FEATURES:
        X_test_corrupted[c] = X_test_corrupted[c] * 1000.0 + 50000.0
    X_test_trans = preproc_clean.transform(X_test)
    _ = preproc_clean.transform(X_test_corrupted)
    X_dev_trans_2 = preproc_clean.transform(X_dev)
    preproc_diff = float(np.max(np.abs(X_dev_trans_1 - X_dev_trans_2)))
    check4_pass = (preproc_diff == 0.0) and (not np.isnan(X_dev_trans_1).any()) and (not np.isnan(X_test_trans).any())

    # Check 5: Verify Transformed Development Matrix has Zero Mean and Unit Variance on Continuous Features
    n_cont = len(CONTINUOUS_FEATURES)
    dev_means_max_abs = float(np.max(np.abs(np.mean(X_dev_trans_1[:, :n_cont], axis=0))))
    dev_stds_max_err = float(np.max(np.abs(np.std(X_dev_trans_1[:, :n_cont], axis=0) - 1.0)))
    # Test set means/stds must NOT be 0 and 1 (proving Test was NOT standardized on its own distribution)
    test_means_abs = float(np.mean(np.abs(np.mean(X_test_trans[:, :n_cont], axis=0))))
    check5_pass = (dev_means_max_abs < 1e-6) and (dev_stds_max_err < 1e-6) and (test_means_abs > 0.05)

    leak_audit_df = pd.DataFrame([
        {
            "Audit_Gate": "1. Retrospective / Target Column Exclusion Gate",
            "Verification_Metric": f"Leaked columns in X: {len(leaked_cols)}",
            "Status": "PASS" if check1_pass else "FAIL",
        },
        {
            "Audit_Gate": "2. Temporal 4-Quarter Purge Gap Enforcement",
            "Verification_Metric": f"Dev max=2008Q4, Test min=2010Q1 (Purged 2009Q1-Q4: {len(gap_pl):,} rows)",
            "Status": "PASS" if check2_pass else "FAIL",
        },
        {
            "Audit_Gate": "3. Future Perturbation Invariance (Causal Window Check)",
            "Verification_Metric": f"Max Dev feature diff when post-2008 data mutated: {max_causal_diff:.2e}",
            "Status": "PASS" if check3_pass else "FAIL",
        },
        {
            "Audit_Gate": "4. Preprocessing Pipeline Fit-on-Train Isolation & Zero-NaN Output",
            "Verification_Metric": f"Max Dev transform diff={preproc_diff:.2e} | NaNs in Dev/Test=0",
            "Status": "PASS" if check4_pass else "FAIL",
        },
        {
            "Audit_Gate": "5. Standardization Moments Verification (Dev Mean=0/Std=1 vs. Test Shift)",
            "Verification_Metric": f"Dev max|mean|={dev_means_max_abs:.2e}, max|std-1|={dev_stds_max_err:.2e} | Test mean|shift|={test_means_abs:.4f}",
            "Status": "PASS" if check5_pass else "FAIL",
        },
    ])
    leak_audit_df.to_csv(out_dir / "p4_leakage_audit_report.csv", index=False)
    print("\n=== 5-POINT ANTI-LEAKAGE GATE AUDIT REPORT ===")
    print(leak_audit_df.to_string(index=False))

    # Save fitted preprocessor for Part 5
    joblib.dump(preproc_clean, work_dir / "fitted_preprocessor_dev.joblib")
    joblib.dump(preproc_clean, out_dir / "fitted_preprocessor_dev.joblib")

    # =========================================================================
    # 3. UNIVARIATE PREDICTIVE POWER & FITTED PARAMETER CATALOG (DEV SET ONLY)
    # =========================================================================
    print("\n3. Computing Univariate Predictive Power (AUROC) & Fitted Parameters on Development Set...")
    winsor = preproc_clean.named_transformers_["num"].named_steps["winsorizer"]
    imputer = preproc_clean.named_transformers_["num"].named_steps["imputer"]
    scaler = preproc_clean.named_transformers_["num"].named_steps["scaler"]

    catalog_rows = []
    for idx, col in enumerate(CONTINUOUS_FEATURES):
        cat_name, desc = FEATURE_META[col]
        raw_vals = X_dev[col].to_numpy(dtype=np.float64)
        miss_pct_dev = float(np.isnan(raw_vals).mean() * 100.0)

        # Compute univariate AUROC on Development set using training-winsorized & median-imputed column
        col_win = np.clip(raw_vals, winsor.lower_bounds_[idx], winsor.upper_bounds_[idx])
        col_imp = np.where(np.isnan(col_win), imputer.statistics_[idx], col_win)

        auc_raw = roc_auc_score(y_dev, col_imp)
        # Direction-agnostic discriminatory power: max(AUC, 1 - AUC)
        auc_disc = max(auc_raw, 1.0 - auc_raw)
        direction = "+" if auc_raw >= 0.5 else "-"

        mean_fail = float(np.mean(col_imp[y_dev == 1]))
        mean_nonfail = float(np.mean(col_imp[y_dev == 0]))

        catalog_rows.append({
            "Feature": col,
            "CAMELS_Group": cat_name,
            "Description": desc,
            "Dev_Missing_%": round(miss_pct_dev, 2),
            "Train_Q01_Bound": round(float(winsor.lower_bounds_[idx]), 3),
            "Train_Median_Impute": round(float(imputer.statistics_[idx]), 3),
            "Train_Q99_Bound": round(float(winsor.upper_bounds_[idx]), 3),
            "Dev_Mean_NonFail_y0": round(mean_nonfail, 3),
            "Dev_Mean_Fail_y1": round(mean_fail, 3),
            "Risk_Direction": direction,
            "Dev_Univariate_AUROC": round(float(auc_disc), 4),
        })

    catalog_df = pd.DataFrame(catalog_rows).sort_values("Dev_Univariate_AUROC", ascending=False)
    catalog_df.to_csv(out_dir / "p4_feature_catalog_and_univariate_auc.csv", index=False)

    ohe_cats = preproc_clean.named_transformers_["cat"].named_steps["ohe"].categories_[0].tolist()
    print(f"\n   Output Matrix Shapes: X_dev_transformed = {X_dev_trans_1.shape} | X_test_transformed = {X_test_trans.shape}")
    print(f"   Breakdown: {len(CONTINUOUS_FEATURES)} continuous + {len(BINARY_FEATURES)} binary + {len(ohe_cats)} one-hot BKCLASS ({ohe_cats}) = {X_dev_trans_1.shape[1]} total columns.")
    print(f"   Completed Part 4 pipeline in {time.perf_counter() - t_start:.2f}s.\n")

    print("=== TOP 20 FEATURES BY DEVELOPMENT-SET UNIVARIATE AUROC (1984Q1-2008Q4) ===")
    print(catalog_df.head(20).to_string(index=False))

    print("\n=== REMAINING 16 CONTINUOUS FEATURES ===")
    print(catalog_df.tail(16).to_string(index=False))


if __name__ == "__main__":
    main()

1. Building Point-in-Time Gold Feature Table from /content/work_p3/panel_silver.parquet...
   Materialized features_gold.parquet: 1,677,817 active rows x 51 cols (406.85 MB)

=== PARTITION & LABEL HORIZON SUMMARY ===
                                split_role    Rows  Distinct_Banks  Positive_Labels_y1  Failed_Banks Min_REPDTE Max_REPDTE  Positive_Rate_%
             1_Development (1984Q1-2008Q4) 1259614           23883               10680          2835   19840331   20081231           0.8479
            2_Purge_Gap_4Q (2009Q1-2009Q4)   32842            8339                 670           276   20090331   20091231           2.0401
                    3_Test (2010Q1-2025Q2)  367860            8116                1109           341   20100331   20250630           0.3015
4_Incomplete_Label_Horizon (2025Q3-2026Q2)   17501            4454                  11             4   20250930   20260630           0.0629

2. Executing Automated 5-Point Anti-Leakage Verification Audit...

=== 5-POINT ANT

## Part 5: Early-Warning Modelling, Cost-Sensitive Thresholding & Out-of-Time Validation
**Objective:** Train, tune, and evaluate machine learning classifiers under extreme class imbalance, leveraging real-world supervisory costs to optimize the alarm threshold.

**Sections:**
1. **3-Fold Purged Expanding-Window Temporal CV (`src/p5_sklearn_models_and_cost_threshold.py`):** Evaluates 4 candidate architectures (Baseline LogReg, Full L2 LogReg, Random Forest, and HistGradientBoosting) inside the Development set. Each fold enforces a strict 4-quarter purge buffer to prevent label-horizon leakage.
2. **Data-Driven Cost Asymmetry Estimation:** Extracts real Deposit Insurance Fund (DIF) resolution losses (`COST`) to estimate Missed Failure Cost ($C_{FN}$) versus targeted on-site examination False Alarm Cost ($C_{FP}$).
3. **Threshold Calibration:** Sweeps probability thresholds on Out-of-Fold predictions to find the optimal $\tau^*$ that minimizes expected supervisory cost.
4. **One-Shot Out-of-Time Test Evaluation:** Freezes $\tau^*$, fits on the full Development set, and evaluates exactly once on the `2010Q1-2025Q2` Test set.

**Pipeline Linkage:** This cell generates the definitive classification metrics (PR-AUC, Recall@Top-1%) that answer research question Q3 regarding the system's operational viability.

In [11]:
"""
p5_sklearn_models_and_cost_threshold.py
Implements Part 5 Modelling & Validation:
1. 3-Fold Purged Expanding-Window Temporal Cross-Validation inside Development Set (1984Q1-2008Q4)
   with fold-isolated preprocessing (zero look-ahead leakage).
2. Compares 4 models:
   - M1: Simple Baseline (2-Factor Capital & Earnings Logistic Regression: eq_to_asset + roa)
   - M2: Regularized Full Logistic Regression (L2 Penalty across all 51 features)
   - M3: Random Forest Classifier (Nonlinear Bagging Tree Ensemble)
   - M4: Histogram-Based Gradient Boosted Trees (HistGradientBoostingClassifier, tuned)
3. Estimates False Alarm Cost (C_FP) and Missed Failure Cost (C_FN) directly from
   Development-period FDIC data (COST and operating scale), optimizing the alarm
   threshold tau* on Out-of-Fold (OOF) Development predictions.
4. Evaluates all models and the frozen threshold tau* ONCE on the Out-of-Time Test Set (2010Q1-2025Q2).
"""
from __future__ import annotations

import gc
import os
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import polars as pl
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

CONTINUOUS_FEATURES = [
    "eq_to_asset", "tier1_lev_ratio", "total_rbc_ratio", "texas_ratio",
    "npl_to_loans", "nonaccrual_to_loans", "pastdue90_to_loans", "pastdue30_89_to_loans",
    "ore_to_asset", "reserves_to_loans", "reserves_to_npl", "net_chargeoffs_to_loans",
    "cost_to_income", "nonii_to_rev", "re_loan_share", "ci_loan_share", "con_loan_share",
    "roa", "roe_clean", "nim", "prov_to_asset", "funding_cost",
    "liquid_to_asset", "loan_to_dep", "brokered_to_dep", "core_dep_to_asset",
    "log_asset", "bank_age_years",
    "asset_growth_yoy", "dep_growth_yoy", "eq_to_asset_chg_4q", "npl_ratio_chg_4q",
    "roa_roll4_mean", "roa_roll4_std",
    "roa_peer_zscore", "state_npl_mean",
]
BINARY_FEATURES = [
    "flag_neg_equity", "flag_insolvent_texas", "flag_missing_rbc", "flag_missing_p3",
]
CATEGORICAL_FEATURES = ["BKCLASS"]
ALL_MODEL_FEATURES = CONTINUOUS_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES
BASELINE_2_FEATURES = ["eq_to_asset", "roa"]


class QuantileWinsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, lower: float = 0.01, upper: float = 0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X: np.ndarray | pd.DataFrame, y=None) -> "QuantileWinsorizer":
        X_arr = np.asarray(X, dtype=np.float64)
        self.lower_bounds_ = np.nanquantile(X_arr, self.lower, axis=0)
        self.upper_bounds_ = np.nanquantile(X_arr, self.upper, axis=0)
        self.n_features_in_ = X_arr.shape[1]
        return self

    def transform(self, X: np.ndarray | pd.DataFrame) -> np.ndarray:
        X_arr = np.asarray(X, dtype=np.float64)
        return np.clip(X_arr, self.lower_bounds_, self.upper_bounds_)


def build_preprocessor(cont_cols: list[str], bin_cols: list[str], cat_cols: list[str]) -> ColumnTransformer:
    transformers = [
        ("num", Pipeline([
            ("winsorizer", QuantileWinsorizer(lower=0.01, upper=0.99)),
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), cont_cols),
    ]
    if bin_cols:
        transformers.append(("bin", SimpleImputer(strategy="constant", fill_value=0.0), bin_cols))
    if cat_cols:
        transformers.append(("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float64)),
        ]), cat_cols))
    return ColumnTransformer(transformers=transformers, remainder="drop")


def recall_at_top_k_pct(y_true: np.ndarray, y_prob: np.ndarray, k_pct: float) -> float:
    """Fraction of all positive failures captured when auditing the top k_pct% highest-risk bank-quarters."""
    n_pos = int(y_true.sum())
    if n_pos == 0:
        return 0.0
    cutoff_n = max(1, int(len(y_true) * (k_pct / 100.0)))
    top_idx = np.argpartition(y_prob, -cutoff_n)[-cutoff_n:]
    return float(y_true[top_idx].sum() / n_pos)


def evaluate_predictions(y_true: np.ndarray, y_prob: np.ndarray) -> dict[str, float]:
    return {
        "PR_AUC": round(float(average_precision_score(y_true, y_prob)), 4),
        "AUROC": round(float(roc_auc_score(y_true, y_prob)), 4),
        "Recall_Top1Pct": round(recall_at_top_k_pct(y_true, y_prob, 1.0), 4),
        "Recall_Top5Pct": round(recall_at_top_k_pct(y_true, y_prob, 5.0), 4),
        "Brier_Score": round(float(brier_score_loss(y_true, y_prob)), 5),
    }


def locate_gold_parquet(data_dir: Path) -> Path:
    candidates = [
        Path("/content/work_p4/features_gold.parquet"),
        data_dir.parent / "data_processed" / "features_gold.parquet",
        data_dir / "data_processed" / "features_gold.parquet",
        Path("/content/drive/MyDrive/BFT65B_Project/data_raw/data_processed/features_gold.parquet"),
        Path("/content/drive/MyDrive/BFT65B_Project/data_processed/features_gold.parquet"),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Cannot locate features_gold.parquet in {candidates}")


def get_models() -> dict[str, tuple[list[str], list[str], list[str], object]]:
    return {
        "M1_Baseline_2Factor_LogReg": (
            BASELINE_2_FEATURES, [], [],
            LogisticRegression(C=1.0, max_iter=300, random_state=RANDOM_SEED),
        ),
        "M2_Full_Logistic_L2": (
            CONTINUOUS_FEATURES, BINARY_FEATURES, CATEGORICAL_FEATURES,
            LogisticRegression(C=0.1, max_iter=500, random_state=RANDOM_SEED),
        ),
        "M3_Random_Forest": (
            CONTINUOUS_FEATURES, BINARY_FEATURES, CATEGORICAL_FEATURES,
            RandomForestClassifier(
                n_estimators=120, max_depth=12, min_samples_leaf=25,
                n_jobs=-1, random_state=RANDOM_SEED,
            ),
        ),
        "M4_HistGradientBoosting": (
            CONTINUOUS_FEATURES, BINARY_FEATURES, CATEGORICAL_FEATURES,
            HistGradientBoostingClassifier(
                learning_rate=0.05, max_iter=180, max_leaf_nodes=31,
                min_samples_leaf=50, l2_regularization=1.0, random_state=RANDOM_SEED,
            ),
        ),
    }


def main() -> None:
    t0_all = time.perf_counter()
    data_dir = Path(os.environ.get("FDIC_DATA_DIR", "data_raw/fdic"))
    out_dir = Path(os.environ.get("FDIC_OUTPUT_DIR", "outputs"))
    out_dir.mkdir(parents=True, exist_ok=True)

    gold_path = locate_gold_parquet(data_dir)
    print(f"1. Loading Gold Feature Table from {gold_path}...")
    df_gold = pl.read_parquet(gold_path)

    dev_df = df_gold.filter(pl.col("split_role") == "1_Development (1984Q1-2008Q4)").to_pandas()
    test_df = df_gold.filter(pl.col("split_role") == "3_Test (2010Q1-2025Q2)").to_pandas()
    del df_gold
    gc.collect()

    print(f"   Development Set (1984Q1-2008Q4): {len(dev_df):,} rows | {int(dev_df['target_fail_4q'].sum()):,} positive labels")
    print(f"   Out-of-Time Test Set (2010Q1-2025Q2): {len(test_df):,} rows | {int(test_df['target_fail_4q'].sum()):,} positive labels")

    # =========================================================================
    # 2. 3-FOLD PURGED EXPANDING-WINDOW TEMPORAL CROSS-VALIDATION (DEV ONLY)
    # =========================================================================
    # Each fold leaves a strict 4-quarter purge buffer between Train end and Val start
    purged_folds = [
        ("Fold_1 (Train 1984-1987 | Purge 1988 | Val 1989-1991)", "19871231", "19890331", "19911231"),
        ("Fold_2 (Train 1984-1991 | Purge 1992 | Val 1993-2000)", "19911231", "19930331", "20001231"),
        ("Fold_3 (Train 1984-2000 | Purge 2001 | Val 2002-2008)", "20001231", "20020331", "20081231"),
    ]

    print("\n2. Running 3-Fold Purged Expanding-Window Temporal Cross-Validation on Development Set...")
    fold_meta_rows = []
    for f_name, tr_end, val_start, val_end in purged_folds:
        tr_mask = dev_df["REPDTE"] <= tr_end
        val_mask = (dev_df["REPDTE"] >= val_start) & (dev_df["REPDTE"] <= val_end)
        fold_meta_rows.append({
            "Fold": f_name,
            "Train_Rows": int(tr_mask.sum()),
            "Train_Pos_y1": int(dev_df.loc[tr_mask, "target_fail_4q"].sum()),
            "Val_Rows": int(val_mask.sum()),
            "Val_Pos_y1": int(dev_df.loc[val_mask, "target_fail_4q"].sum()),
            "Val_Base_Rate_%": round(float(dev_df.loc[val_mask, "target_fail_4q"].mean() * 100), 4),
        })
    fold_meta_df = pd.DataFrame(fold_meta_rows)
    print(fold_meta_df.to_string(index=False))

    # Hyperparameter grid check for HistGradientBoosting on Purged CV before locking M4
    hgb_grid = [
        {"learning_rate": 0.03, "max_leaf_nodes": 15, "l2_regularization": 2.0},
        {"learning_rate": 0.05, "max_leaf_nodes": 31, "l2_regularization": 1.0},
        {"learning_rate": 0.08, "max_leaf_nodes": 63, "l2_regularization": 0.5},
    ]
    print("\n   Tuning HistGradientBoosting hyperparameters via Purged Temporal CV (PR-AUC):")
    best_hgb_params = hgb_grid[1]
    best_hgb_cv_ap = -1.0
    for params in hgb_grid:
        ap_scores = []
        for _, tr_end, val_start, val_end in purged_folds:
            tr_sub = dev_df[dev_df["REPDTE"] <= tr_end]
            val_sub = dev_df[(dev_df["REPDTE"] >= val_start) & (dev_df["REPDTE"] <= val_end)]
            prep = build_preprocessor(CONTINUOUS_FEATURES, BINARY_FEATURES, CATEGORICAL_FEATURES)
            X_tr = prep.fit_transform(tr_sub[ALL_MODEL_FEATURES])
            X_va = prep.transform(val_sub[ALL_MODEL_FEATURES])
            clf = HistGradientBoostingClassifier(
                max_iter=180, min_samples_leaf=50, random_state=RANDOM_SEED, **params
            )
            clf.fit(X_tr, tr_sub["target_fail_4q"].to_numpy())
            p_va = clf.predict_proba(X_va)[:, 1]
            ap_scores.append(average_precision_score(val_sub["target_fail_4q"].to_numpy(), p_va))
        mean_ap = float(np.mean(ap_scores))
        print(f"      Params {params} -> Mean Purged CV PR-AUC = {mean_ap:.4f}")
        if mean_ap > best_hgb_cv_ap:
            best_hgb_cv_ap = mean_ap
            best_hgb_params = params
    print(f"   Selected HistGradientBoosting Params: {best_hgb_params} (Mean CV PR-AUC = {best_hgb_cv_ap:.4f})")

    # Evaluate all 4 candidate models across Purged Expanding-Window CV & collect OOF predictions
    models_dict = get_models()
    models_dict["M4_HistGradientBoosting"] = (
        CONTINUOUS_FEATURES, BINARY_FEATURES, CATEGORICAL_FEATURES,
        HistGradientBoostingClassifier(
            max_iter=180, min_samples_leaf=50, random_state=RANDOM_SEED, **best_hgb_params
        ),
    )

    cv_summary_rows = []
    oof_mask_total = np.zeros(len(dev_df), dtype=bool)
    oof_preds = {m_name: np.zeros(len(dev_df), dtype=np.float64) for m_name in models_dict}

    for m_name, (c_cols, b_cols, cat_cols, base_est) in models_dict.items():
        t_m = time.perf_counter()
        fold_metrics = []
        feat_list = c_cols + b_cols + cat_cols

        for f_name, tr_end, val_start, val_end in purged_folds:
            tr_idx = dev_df["REPDTE"] <= tr_end
            val_idx = (dev_df["REPDTE"] >= val_start) & (dev_df["REPDTE"] <= val_end)
            oof_mask_total |= val_idx.to_numpy()

            prep = build_preprocessor(c_cols, b_cols, cat_cols)
            X_tr = prep.fit_transform(dev_df.loc[tr_idx, feat_list])
            y_tr = dev_df.loc[tr_idx, "target_fail_4q"].to_numpy()
            X_va = prep.transform(dev_df.loc[val_idx, feat_list])
            y_va = dev_df.loc[val_idx, "target_fail_4q"].to_numpy()

            from sklearn.base import clone
            clf = clone(base_est)
            clf.fit(X_tr, y_tr)
            prob_va = clf.predict_proba(X_va)[:, 1]
            oof_preds[m_name][val_idx.to_numpy()] = prob_va

            m_eval = evaluate_predictions(y_va, prob_va)
            fold_metrics.append(m_eval)

        # Compute Pooled Out-of-Fold (OOF) Metrics across the 3 disjoint validation windows
        y_oof = dev_df.loc[oof_mask_total, "target_fail_4q"].to_numpy()
        p_oof = oof_preds[m_name][oof_mask_total]
        pooled_eval = evaluate_predictions(y_oof, p_oof)

        cv_summary_rows.append({
            "Model": m_name,
            "Fold1_PR_AUC": fold_metrics[0]["PR_AUC"],
            "Fold2_PR_AUC": fold_metrics[1]["PR_AUC"],
            "Fold3_PR_AUC": fold_metrics[2]["PR_AUC"],
            "Mean_Fold_PR_AUC": round(float(np.mean([f["PR_AUC"] for f in fold_metrics])), 4),
            "Pooled_OOF_PR_AUC": pooled_eval["PR_AUC"],
            "Pooled_OOF_AUROC": pooled_eval["AUROC"],
            "Pooled_OOF_Recall_Top1Pct": pooled_eval["Recall_Top1Pct"],
            "Pooled_OOF_Recall_Top5Pct": pooled_eval["Recall_Top5Pct"],
            "Pooled_OOF_Brier": pooled_eval["Brier_Score"],
            "CV_Elapsed_s": round(time.perf_counter() - t_m, 2),
        })

    cv_df = pd.DataFrame(cv_summary_rows)
    cv_df.to_csv(out_dir / "p5_purged_cv_model_comparison.csv", index=False)
    print("\n=== 3-FOLD PURGED EXPANDING-WINDOW CV RESULTS (DEVELOPMENT SET) ===")
    print(cv_df.to_string(index=False))

    # =========================================================================
    # 3. DATA-DRIVEN COST ESTIMATION & THRESHOLD OPTIMIZATION (DEV OOF ONLY)
    # =========================================================================
    print("\n3. Estimating Asymmetric Supervisory Costs from Development Data (1984Q1-2008Q4)...")
    # Estimate DIF resolution cost from distinct failed banks in Development set with valid COST > 0
    dev_failed_rows = dev_df[(dev_df["target_fail_4q"] == 1) & (dev_df["next_fail_cost"] > 0)]
    dev_failed_banks = dev_failed_rows.groupby("CERT").agg(
        dif_cost_k=("next_fail_cost", "first"),
        asset_k=("ASSET", "mean"),
    )
    median_dif_cost_k = float(dev_failed_banks["dif_cost_k"].median())
    trimmed_mean_dif_cost_k = float(
        dev_failed_banks["dif_cost_k"].clip(upper=dev_failed_banks["dif_cost_k"].quantile(0.95)).mean()
    )
    median_fail_asset_k = float(dev_failed_banks["asset_k"].median())

    # Early supervisory intervention (1-4 quarters prior to failure) mitigates a fraction alpha_q = 10%
    # of the trimmed-mean DIF resolution loss per bank-quarter warning (or 25% of median DIF loss).
    # Let us anchor C_FN and C_FP transparently in thousands of USD:
    # C_FN: Avoidable DIF resolution loss per missed failing bank-quarter = 25% of Median DIF Loss (or ~5% of Trimmed Mean)
    c_fn_k = round(0.05 * trimmed_mean_dif_cost_k, 2)  # in $ thousands
    # C_FP: Direct cost of an intensive targeted FDIC on-site safety-and-soundness examination
    # Estimated from bank scale (~0.25% of median failed bank assets, or ~$150k-$200k per targeted exam)
    c_fp_k = round(0.0025 * median_fail_asset_k, 2)
    cost_ratio = round(c_fn_k / c_fp_k, 2)
    bayes_theoretical_tau = round(1.0 / (1.0 + cost_ratio), 4)

    print(f"   Empirical Dev Failed Banks (with DIF COST > 0): {len(dev_failed_banks):,} banks")
    print(f"   Dev Median DIF Resolution Cost (COST)         : ${median_dif_cost_k:,.1f}k | 95%-Trimmed Mean: ${trimmed_mean_dif_cost_k:,.1f}k")
    print(f"   Dev Median Failed Bank Asset Scale (ASSET)    : ${median_fail_asset_k:,.1f}k")
    print(f"   Estimated Missed Failure Cost per Quarter C_FN: ${c_fn_k:,.2f}k (5% avoidable DIF loss per quarter)")
    print(f"   Estimated False Alarm Audit Cost C_FP         : ${c_fp_k:,.2f}k (0.25% of median bank assets per on-site exam)")
    print(f"   Empirical Cost Asymmetry Ratio (C_FN / C_FP)  : {cost_ratio:.2f} : 1 (Bayes theoretical threshold = {bayes_theoretical_tau:.4f})")

    # Sweep thresholds tau in [0.005, 0.90] on Champion Model (M4_HistGradientBoosting) OOF predictions
    y_oof_all = dev_df.loc[oof_mask_total, "target_fail_4q"].to_numpy()
    p_oof_m4 = oof_preds["M4_HistGradientBoosting"][oof_mask_total]
    n_oof = len(y_oof_all)
    n_pos_oof = int(y_oof_all.sum())
    no_model_cost_per_1k = (n_pos_oof * c_fn_k) / (n_oof / 1000.0)

    thresholds = np.linspace(0.005, 0.80, 160)
    sweep_rows = []
    best_tau = 0.10
    min_cost_per_1k = float("inf")

    for tau in thresholds:
        pred_bin = (p_oof_m4 >= tau).astype(np.int8)
        tn, fp, fn, tp = confusion_matrix(y_oof_all, pred_bin).ravel()
        total_cost_k = fn * c_fn_k + fp * c_fp_k
        cost_per_1k = total_cost_k / (n_oof / 1000.0)
        savings_pct = (1.0 - cost_per_1k / no_model_cost_per_1k) * 100.0
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

        if cost_per_1k < min_cost_per_1k:
            min_cost_per_1k = cost_per_1k
            best_tau = round(float(tau), 4)

        sweep_rows.append({
            "Threshold_tau": round(float(tau), 4),
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "F1_Score": round(f1, 4),
            "Expected_Cost_per_1k_BankQuarters_$k": round(cost_per_1k, 2),
            "Cost_Savings_vs_NoAudit_%": round(savings_pct, 2),
        })

    sweep_df = pd.DataFrame(sweep_rows)
    sweep_df.to_csv(out_dir / "p5_dev_oof_threshold_cost_curve.csv", index=False)

    # Extract representative thresholds for reporting table
    rep_taus = sorted(list({0.01, 0.025, 0.05, best_tau, 0.10, 0.15, 0.20, 0.30, 0.50}))
    rep_rows = []
    for t_val in rep_taus:
        idx = (sweep_df["Threshold_tau"] - t_val).abs().idxmin()
        rep_rows.append(sweep_df.loc[idx].to_dict())
    rep_df = pd.DataFrame(rep_rows).drop_duplicates(subset=["Threshold_tau"])
    print(f"\n=== DEVELOPMENT OOF COST-THRESHOLD OPTIMIZATION (OPTIMAL tau* = {best_tau:.4f}) ===")
    print(rep_df.to_string(index=False))

    # =========================================================================
    # 4. SINGLE ONE-SHOT EVALUATION ON OUT-OF-TIME TEST SET (2010Q1-2025Q2)
    # =========================================================================
    print(f"\n4. Refitting Models on Full Development Set (1984Q1-2008Q4) and Evaluating ONCE on Test Set (2010Q1-2025Q2) at Frozen tau* = {best_tau:.4f}...")
    y_dev_full = dev_df["target_fail_4q"].to_numpy()
    y_test_full = test_df["target_fail_4q"].to_numpy()
    n_test = len(y_test_full)
    n_pos_test = int(y_test_full.sum())
    test_no_audit_cost_per_1k = (n_pos_test * c_fn_k) / (n_test / 1000.0)

    test_results = []
    test_probs_store = {}
    fitted_models_store = {}

    for m_name, (c_cols, b_cols, cat_cols, base_est) in models_dict.items():
        t_fit = time.perf_counter()
        feat_list = c_cols + b_cols + cat_cols
        prep = build_preprocessor(c_cols, b_cols, cat_cols)

        X_dev_mat = prep.fit_transform(dev_df[feat_list])
        X_test_mat = prep.transform(test_df[feat_list])

        from sklearn.base import clone
        clf = clone(base_est)
        clf.fit(X_dev_mat, y_dev_full)
        fit_time_s = round(time.perf_counter() - t_fit, 2)

        prob_test = clf.predict_proba(X_test_mat)[:, 1]
        test_probs_store[m_name] = prob_test
        fitted_models_store[m_name] = (prep, clf)

        rank_metrics = evaluate_predictions(y_test_full, prob_test)

        # Evaluate at frozen cost-optimal threshold best_tau (and compare against naive 0.50)
        pred_opt = (prob_test >= best_tau).astype(np.int8)
        tn, fp, fn, tp = confusion_matrix(y_test_full, pred_opt).ravel()
        cost_opt_per_1k = (fn * c_fn_k + fp * c_fp_k) / (n_test / 1000.0)
        savings_opt_pct = (1.0 - cost_opt_per_1k / test_no_audit_cost_per_1k) * 100.0

        pred_50 = (prob_test >= 0.50).astype(np.int8)
        _, fp50, fn50, tp50 = confusion_matrix(y_test_full, pred_50).ravel()
        rec_50 = tp50 / (tp50 + fn50) if (tp50 + fn50) > 0 else 0.0

        # Bank-level detection rate across the 341 distinct failed banks in Test set
        test_eval_frame = pd.DataFrame({
            "CERT": test_df["CERT"].to_numpy(),
            "q_idx": test_df["q_idx"].to_numpy(),
            "y": y_test_full,
            "pred": pred_opt,
        })
        fail_windows = test_eval_frame[test_eval_frame["y"] == 1]
        banks_detected = int(fail_windows.groupby("CERT")["pred"].max().sum())
        total_test_failed_banks = int(fail_windows["CERT"].nunique())
        bank_detect_rate = round(banks_detected / total_test_failed_banks * 100.0, 2)

        test_results.append({
            "Model": m_name,
            "Test_PR_AUC": rank_metrics["PR_AUC"],
            "Test_AUROC": rank_metrics["AUROC"],
            "Test_Recall_Top1Pct": rank_metrics["Recall_Top1Pct"],
            "Test_Recall_Top5Pct": rank_metrics["Recall_Top5Pct"],
            "Test_Brier": rank_metrics["Brier_Score"],
            "Recall_at_Naive_0.50": round(rec_50, 4),
            "Precision_at_tau*": round(float(precision_score(y_test_full, pred_opt, zero_division=0)), 4),
            "Recall_at_tau*": round(float(recall_score(y_test_full, pred_opt, zero_division=0)), 4),
            "F1_at_tau*": round(float(f1_score(y_test_full, pred_opt, zero_division=0)), 4),
            "TP_at_tau*": int(tp),
            "FP_at_tau*": int(fp),
            "FN_at_tau*": int(fn),
            "Test_Cost_per_1k_$k": round(cost_opt_per_1k, 2),
            "Test_Cost_Savings_%": round(savings_opt_pct, 2),
            "Failed_Banks_Caught_341": f"{banks_detected}/{total_test_failed_banks} ({bank_detect_rate}%)",
            "Fit_Time_s": fit_time_s,
        })

    test_res_df = pd.DataFrame(test_results)
    test_res_df.to_csv(out_dir / "p5_test_set_final_evaluation.csv", index=False)
    print("\n=== ONE-SHOT OUT-OF-TIME TEST SET EVALUATION (2010Q1-2025Q2, N = 367,860, 341 FAILED BANKS) ===")
    print(test_res_df.to_string(index=False))

    # =========================================================================
    # 5. SUB-PERIOD REGIME ANALYSIS ON TEST SET (2010-2014 GFC AFTERMATH VS 2015-2025)
    # =========================================================================
    test_df["prob_M4"] = test_probs_store["M4_HistGradientBoosting"]
    test_df["pred_M4_opt"] = (test_df["prob_M4"] >= best_tau).astype(np.int8)

    sub_periods = [
        ("Post-GFC Failure Wave (2010Q1-2014Q4)", "20100331", "20141231"),
        ("Modern & 2023 Liquidity Crisis Regime (2015Q1-2025Q2)", "20150331", "20250630"),
    ]
    sub_rows = []
    for sp_name, s_start, s_end in sub_periods:
        sub = test_df[(test_df["REPDTE"] >= s_start) & (test_df["REPDTE"] <= s_end)]
        y_s = sub["target_fail_4q"].to_numpy()
        p_s = sub["prob_M4"].to_numpy()
        pr_s = (p_s >= best_tau).astype(np.int8)
        m_s = evaluate_predictions(y_s, p_s)
        f_sub = sub[sub["target_fail_4q"] == 1]
        b_caught = int(f_sub.groupby("CERT")["pred_M4_opt"].max().sum())
        b_tot = int(f_sub["CERT"].nunique())
        sub_rows.append({
            "Test_SubPeriod": sp_name,
            "Rows": len(sub),
            "Pos_Labels_y1": int(y_s.sum()),
            "Failed_Banks": b_tot,
            "PR_AUC": m_s["PR_AUC"],
            "AUROC": m_s["AUROC"],
            "Recall_Top1Pct": m_s["Recall_Top1Pct"],
            "Recall_Top5Pct": m_s["Recall_Top5Pct"],
            "Recall_at_tau*": round(float(recall_score(y_s, pr_s, zero_division=0)), 4),
            "Precision_at_tau*": round(float(precision_score(y_s, pr_s, zero_division=0)), 4),
            "Banks_Caught": f"{b_caught}/{b_tot} ({b_caught/b_tot*100:.1f}%)",
        })
    sub_df = pd.DataFrame(sub_rows)
    sub_df.to_csv(out_dir / "p5_test_regime_subperiods.csv", index=False)
    print("\n=== CHAMPION MODEL (M4) PERFORMANCE ACROSS TEST SUB-REGIMES ===")
    print(sub_df.to_string(index=False))

    print(f"\nCompleted Cell 1 (scikit-learn CV, Cost Thresholding, and One-Shot Test Evaluation) in {time.perf_counter() - t0_all:.2f}s.")


if __name__ == "__main__":
    main()

1. Loading Gold Feature Table from /content/work_p4/features_gold.parquet...
   Development Set (1984Q1-2008Q4): 1,259,614 rows | 10,680 positive labels
   Out-of-Time Test Set (2010Q1-2025Q2): 367,860 rows | 1,109 positive labels

2. Running 3-Fold Purged Expanding-Window Temporal Cross-Validation on Development Set...
                                                 Fold  Train_Rows  Train_Pos_y1  Val_Rows  Val_Pos_y1  Val_Base_Rate_%
Fold_1 (Train 1984-1987 | Purge 1988 | Val 1989-1991)      285674          3655    186447        3533           1.8949
Fold_2 (Train 1984-1991 | Purge 1992 | Val 1993-2000)      539576          9518    371827         240           0.0645
Fold_3 (Train 1984-2000 | Purge 2001 | Val 2002-2008)      968224         10144    252096         499           0.1979

   Tuning HistGradientBoosting hyperparameters via Purged Temporal CV (PR-AUC):


/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [7]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [7]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [7]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/py

      Params {'learning_rate': 0.03, 'max_leaf_nodes': 15, 'l2_regularization': 2.0} -> Mean Purged CV PR-AUC = 0.3687


/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [7]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [7]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [7]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/py

      Params {'learning_rate': 0.05, 'max_leaf_nodes': 31, 'l2_regularization': 1.0} -> Mean Purged CV PR-AUC = 0.3689


/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [7]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [7]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [7]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/py

      Params {'learning_rate': 0.08, 'max_leaf_nodes': 63, 'l2_regularization': 0.5} -> Mean Purged CV PR-AUC = 0.3671
   Selected HistGradientBoosting Params: {'learning_rate': 0.05, 'max_leaf_nodes': 31, 'l2_regularization': 1.0} (Mean CV PR-AUC = 0.3689)


/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [7]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [7]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [7]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/py


=== 3-FOLD PURGED EXPANDING-WINDOW CV RESULTS (DEVELOPMENT SET) ===
                     Model  Fold1_PR_AUC  Fold2_PR_AUC  Fold3_PR_AUC  Mean_Fold_PR_AUC  Pooled_OOF_PR_AUC  Pooled_OOF_AUROC  Pooled_OOF_Recall_Top1Pct  Pooled_OOF_Recall_Top5Pct  Pooled_OOF_Brier  CV_Elapsed_s
M1_Baseline_2Factor_LogReg        0.3564        0.0836        0.2049            0.2150             0.2895            0.9614                     0.6166                     0.9068           0.00466          4.68
       M2_Full_Logistic_L2        0.4504        0.1131        0.3006            0.2880             0.3066            0.9470                     0.5325                     0.7706           0.00429         44.11
          M3_Random_Forest        0.5355        0.2558        0.3011            0.3641             0.4642            0.9814                     0.6655                     0.9380           0.00374       1784.61
   M4_HistGradientBoosting        0.5485        0.2550        0.3031            0.3689     

### 5.2. Distributed PySpark MLlib Pipeline Demo
**Objective:** Fulfill the requirement to demonstrate modeling capabilities using a distributed framework.

* Constructs a `pyspark.ml.Pipeline` (`Imputer` $\rightarrow$ `StringIndexer` $\rightarrow$ `OneHotEncoder` $\rightarrow$ `VectorAssembler` $\rightarrow$ `StandardScaler`).
* Standardizes empty strings (`""`) to `"UNKNOWN"` to satisfy Spark MLlib's strict `NominalAttribute` metadata constraints.
* Fits distributed `LogisticRegression` and `GBTClassifier` purely on the Development partition and evaluates PR-AUC on the Test partition.

In [13]:
"""
p5_spark_mllib_demo.py
Implements a distributed PySpark MLlib Pipeline (Imputer -> StringIndexer ->
OneHotEncoder -> VectorAssembler -> StandardScaler -> LogisticRegression & GBTClassifier)
fitted strictly on the Development period (1984Q1-2008Q4) and evaluated on the
Out-of-Time Test period (2010Q1-2025Q2).
Resolves Spark MLlib NominalAttribute empty-string metadata requirement on BKCLASS.
"""
from __future__ import annotations

import gc
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from pyspark.ml import Pipeline as SparkPipeline
from pyspark.ml.classification import GBTClassifier, LogisticRegression as SparkLogisticRegression
from pyspark.ml.feature import (
    Imputer as SparkImputer,
    OneHotEncoder as SparkOHE,
    StandardScaler as SparkStandardScaler,
    StringIndexer,
    VectorAssembler,
)
from pyspark.ml.functions import vector_to_array
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

CONTINUOUS_FEATURES = [
    "eq_to_asset", "tier1_lev_ratio", "total_rbc_ratio", "texas_ratio",
    "npl_to_loans", "nonaccrual_to_loans", "pastdue90_to_loans", "pastdue30_89_to_loans",
    "ore_to_asset", "reserves_to_loans", "reserves_to_npl", "net_chargeoffs_to_loans",
    "cost_to_income", "nonii_to_rev", "re_loan_share", "ci_loan_share", "con_loan_share",
    "roa", "roe_clean", "nim", "prov_to_asset", "funding_cost",
    "liquid_to_asset", "loan_to_dep", "brokered_to_dep", "core_dep_to_asset",
    "log_asset", "bank_age_years",
    "asset_growth_yoy", "dep_growth_yoy", "eq_to_asset_chg_4q", "npl_ratio_chg_4q",
    "roa_roll4_mean", "roa_roll4_std",
    "roa_peer_zscore", "state_npl_mean",
]
BINARY_FEATURES = [
    "flag_neg_equity", "flag_insolvent_texas", "flag_missing_rbc", "flag_missing_p3",
]


def get_spark() -> SparkSession:
    return (
        SparkSession.builder
        .appName("BFT65B-Part5-SparkMLlib-Demo")
        .master("local[*]")
        .config("spark.driver.memory", "8g")
        .config("spark.sql.shuffle.partitions", "16")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")
        .getOrCreate()
    )


def locate_gold_parquet(data_dir: Path) -> Path:
    candidates = [
        Path("/content/work_p4/features_gold.parquet"),
        data_dir.parent / "data_processed" / "features_gold.parquet",
        data_dir / "data_processed" / "features_gold.parquet",
        Path("/content/drive/MyDrive/BFT65B_Project/data_raw/data_processed/features_gold.parquet"),
        Path("/content/drive/MyDrive/BFT65B_Project/data_processed/features_gold.parquet"),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Cannot locate features_gold.parquet in {candidates}")


def recall_at_top_k_pct(y_true: np.ndarray, y_prob: np.ndarray, k_pct: float) -> float:
    n_pos = int(y_true.sum())
    if n_pos == 0:
        return 0.0
    cutoff_n = max(1, int(len(y_true) * (k_pct / 100.0)))
    top_idx = np.argpartition(y_prob, -cutoff_n)[-cutoff_n:]
    return float(y_true[top_idx].sum() / n_pos)


def main() -> None:
    data_dir = Path(os.environ.get("FDIC_DATA_DIR", "data_raw/fdic"))
    out_dir = Path(os.environ.get("FDIC_OUTPUT_DIR", "outputs"))
    out_dir.mkdir(parents=True, exist_ok=True)

    spark = get_spark()
    spark.sparkContext.setLogLevel("ERROR")

    gold_path = locate_gold_parquet(data_dir)
    print(f"1. Loading {gold_path} into PySpark DataFrame...")
    sdf = spark.read.parquet(str(gold_path))

    # Normalize IEEE NaN to NULL, sanitize empty-string BKCLASS for Spark MLlib NominalAttribute, and cast label
    for c in CONTINUOUS_FEATURES:
        sdf = sdf.withColumn(c, F.when(F.isnan(F.col(c)), F.lit(None)).otherwise(F.col(c).cast("double")))
    for b in BINARY_FEATURES:
        sdf = sdf.withColumn(b, F.coalesce(F.col(b).cast("double"), F.lit(0.0)))

    sdf = sdf.withColumn(
        "BKCLASS",
        F.when(
            F.col("BKCLASS").isNull() | (F.trim(F.col("BKCLASS")) == ""),
            F.lit("UNKNOWN"),
        ).otherwise(F.trim(F.col("BKCLASS"))),
    )
    sdf = sdf.withColumn("label", F.col("target_fail_4q").cast("double"))

    # Split into Development (1984Q1-2008Q4) and Out-of-Time Test (2010Q1-2025Q2)
    sdf_dev = sdf.filter(F.col("split_role") == "1_Development (1984Q1-2008Q4)")
    sdf_test = sdf.filter(F.col("split_role") == "3_Test (2010Q1-2025Q2)")

    # Fit training-only 1%-99% winsorization bounds on Development set via approxQuantile
    print("2. Computing Training-Only Quantile Bounds (q01, q99) in Spark on Development Set...")
    quantiles_list = sdf_dev.stat.approxQuantile(CONTINUOUS_FEATURES, [0.01, 0.99], 0.001)
    for idx, c in enumerate(CONTINUOUS_FEATURES):
        q01, q99 = quantiles_list[idx]
        sdf_dev = sdf_dev.withColumn(
            c,
            F.when(F.col(c).isNull(), F.lit(None))
             .when(F.col(c) < q01, F.lit(float(q01)))
             .when(F.col(c) > q99, F.lit(float(q99)))
             .otherwise(F.col(c)),
        )
        sdf_test = sdf_test.withColumn(
            c,
            F.when(F.col(c).isNull(), F.lit(None))
             .when(F.col(c) < q01, F.lit(float(q01)))
             .when(F.col(c) > q99, F.lit(float(q99)))
             .otherwise(F.col(c)),
        )

    sdf_dev.cache()
    sdf_test.cache()
    print(f"   Cached Spark Partitions: Dev = {sdf_dev.count():,} rows | Test = {sdf_test.count():,} rows")

    # Build Spark MLlib Preprocessing Stages (Fitted strictly on sdf_dev)
    imputed_cols = [f"{c}_imp" for c in CONTINUOUS_FEATURES]
    imputer = SparkImputer(
        strategy="median",
        inputCols=CONTINUOUS_FEATURES,
        outputCols=imputed_cols,
    )
    bk_indexer = StringIndexer(inputCol="BKCLASS", outputCol="BKCLASS_idx", handleInvalid="keep")
    bk_ohe = SparkOHE(inputCols=["BKCLASS_idx"], outputCols=["BKCLASS_ohe"], handleInvalid="keep")

    raw_assembler = VectorAssembler(
        inputCols=imputed_cols + BINARY_FEATURES + ["BKCLASS_ohe"],
        outputCol="raw_features",
        handleInvalid="keep",
    )
    scaler = SparkStandardScaler(
        inputCol="raw_features",
        outputCol="features",
        withMean=True,
        withStd=True,
    )

    # Fit shared preprocessing pipeline once on Development data
    t_prep = time.perf_counter()
    prep_pipeline = SparkPipeline(stages=[imputer, bk_indexer, bk_ohe, raw_assembler, scaler])
    prep_model = prep_pipeline.fit(sdf_dev)
    dev_prepared = prep_model.transform(sdf_dev).select("CERT", "REPDTE", "label", "features").cache()
    test_prepared = prep_model.transform(sdf_test).select("CERT", "REPDTE", "label", "features").cache()
    _ = dev_prepared.count()
    _ = test_prepared.count()
    prep_time_s = round(time.perf_counter() - t_prep, 2)
    print(f"   Fitted Spark MLlib Preprocessor on Development Set in {prep_time_s:.2f}s.")

    spark_models = [
        (
            "Spark_MLlib_LogisticRegression (L2)",
            SparkLogisticRegression(
                featuresCol="features", labelCol="label",
                regParam=0.01, elasticNetParam=0.0, maxIter=100,
            ),
        ),
        (
            "Spark_MLlib_GBTClassifier (Depth=5, Iter=30)",
            GBTClassifier(
                featuresCol="features", labelCol="label",
                maxDepth=5, maxIter=30, stepSize=0.1, seed=42,
            ),
        ),
    ]

    mllib_rows = []
    for m_name, estimator in spark_models:
        print(f"3. Fitting {m_name} on Development Set...")
        t0 = time.perf_counter()
        fitted_m = estimator.fit(dev_prepared)
        fit_s = round(time.perf_counter() - t0, 2)

        t1 = time.perf_counter()
        preds_sdf = fitted_m.transform(test_prepared)
        preds_arr_df = (
            preds_sdf
            .select(
                F.col("label").alias("y_true"),
                vector_to_array(F.col("probability")).getItem(1).alias("y_prob"),
            )
            .toPandas()
        )
        infer_s = round(time.perf_counter() - t1, 2)

        y_true = preds_arr_df["y_true"].to_numpy()
        y_prob = preds_arr_df["y_prob"].to_numpy()

        mllib_rows.append({
            "Engine_Model": m_name,
            "Test_PR_AUC": round(float(average_precision_score(y_true, y_prob)), 4),
            "Test_AUROC": round(float(roc_auc_score(y_true, y_prob)), 4),
            "Test_Recall_Top1Pct": round(recall_at_top_k_pct(y_true, y_prob, 1.0), 4),
            "Test_Recall_Top5Pct": round(recall_at_top_k_pct(y_true, y_prob, 5.0), 4),
            "Test_Brier": round(float(brier_score_loss(y_true, y_prob)), 5),
            "Spark_Prep_s": prep_time_s,
            "Model_Fit_s": fit_s,
            "Test_Infer_s": infer_s,
        })

    mllib_df = pd.DataFrame(mllib_rows)
    mllib_df.to_csv(out_dir / "p5_spark_mllib_demo_results.csv", index=False)
    print("\n=== PYSPARK MLLIB DISTRIBUTED PIPELINE DEMO RESULTS (TEST SET 2010Q1-2025Q2) ===")
    print(mllib_df.to_string(index=False))

    sdf_dev.unpersist()
    sdf_test.unpersist()
    dev_prepared.unpersist()
    test_prepared.unpersist()
    gc.collect()


if __name__ == "__main__":
    main()

1. Loading /content/work_p4/features_gold.parquet into PySpark DataFrame...
2. Computing Training-Only Quantile Bounds (q01, q99) in Spark on Development Set...
   Cached Spark Partitions: Dev = 1,259,614 rows | Test = 367,860 rows
   Fitted Spark MLlib Preprocessor on Development Set in 79.81s.
3. Fitting Spark_MLlib_LogisticRegression (L2) on Development Set...
3. Fitting Spark_MLlib_GBTClassifier (Depth=5, Iter=30) on Development Set...

=== PYSPARK MLLIB DISTRIBUTED PIPELINE DEMO RESULTS (TEST SET 2010Q1-2025Q2) ===
                                Engine_Model  Test_PR_AUC  Test_AUROC  Test_Recall_Top1Pct  Test_Recall_Top5Pct  Test_Brier  Spark_Prep_s  Model_Fit_s  Test_Infer_s
         Spark_MLlib_LogisticRegression (L2)       0.5564      0.9780               0.8629               0.9486     0.00189         79.81        41.52          2.76
Spark_MLlib_GBTClassifier (Depth=5, Iter=30)       0.5286      0.9803               0.8810               0.9531     0.00315         79.81       